# V8 — DINOv3 ViT-7B frozen-cache R/O specialist

## 1. Objective and locked constraints

V8 is a cache-only, train-only decision experiment. It consumes the validated V7 cache from `facebook/dinov3-vit7b16-pretrain-lvd1689m`; it never imports, loads, or forwards the ViT-7B backbone. Official raw labels, original-view visual features, seed 42, Macro F1, and the established five folds remain fixed.

The official test set, templates, submissions, external labels, pseudo-labels, relabeling, image metadata, filenames, paths, dimensions, and directory sources are forbidden as predictive inputs. The notebook creates no submission.

## 2. Evidence reviewed and V8 approach decision

The repository review covered the V7 run/decision/selection manifests, all fold-0 head metrics, histories, reports and predictions, both historical assessment prediction tables and reports, per-fold assessment metrics, cache/model/preflight audits, error CSVs and the R/O image gallery, plus V5/V6 notebooks, the competition guide, handoff notes, the DINOv3 report, frozen-readout/layer-selection references, and the waste-domain review.

Verified V7 evidence identifies a narrow residual bottleneck: the selected regularized head made 61 fold-0 errors, 54 of them Recyclable↔Organic; on historical folds 1–4 it made 226 errors, 187 Recyclable↔Organic. Electronic F1 was already 0.995589 on fold 0 and 0.993859 on folds 1–4. The generic last-four-layer weighted-gated head scored 0.988073, below the selected final-layer regularized MLP at 0.989918. Calibration alone cannot improve argmax Macro F1, and the gallery contains genuinely ambiguous/manual-review cases that do not justify automatic relabeling.

Decision: train a small binary Recyclable/Organic specialist on the cached final-layer `concat(CLS, mean(true patches))` representation. Fuse it hierarchically by redistributing only the main head's R/O probability mass, while preserving both the main Electronic probability and its Electronic/non-Electronic decision mask. The primary fusion uses fixed specialist weight 0.75; full replacement at 1.0 is the sole mechanism ablation. Both were fixed before fold-0 scoring.

## 3. Imports and deterministic configuration

All thresholds, schemas, model definitions, fusion rules, and training hyperparameters are declared once below. V7 checkpoint metadata is later required to confirm the control recipe before training.

In [1]:
import gc
import hashlib
import json
import math
import os
import platform
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, log_loss
from torch.utils.data import DataLoader, Dataset

NOTEBOOK_STARTED_AT = time.perf_counter()
EXPERIMENT_NAME = 'v8_dinov3_vit7b_frozen_cache_upgrade'
V8_SCHEMA_VERSION = 1
V8_REVISION_MARKER = 'v8_ro_hierarchical_specialist_final_cache_v1'
V7_DIRECTORY_NAME = 'v7_dinov3_vit7b_frozen_raw'
V7_REVISION_MARKER = 'v7_official_final_norm_before_pooling_v2'
MODEL_ID = 'facebook/dinov3-vit7b16-pretrain-lvd1689m'
EXPECTED_MODEL_REVISION = 'b80367753773648a6793235ab9c65cdbb029506f'
EXPECTED_CACHE_SIGNATURE = 'c722dc39a8cba286d3aba34d88f3e8f87071353bc6585e755873659de8560ddc'
EXPECTED_FOLD_ASSIGNMENT_SHA256 = '0a06976e5dc3cdb0b3f1f5043ff033bf95a40dd2b0cb82154bbf34d701ee305c'
EXPECTED_ORDERED_IMAGE_IDS_SHA256 = '210f8f14797be49bad962e81f1da660a257ec28ee2136b54a7516950a66cb2e8'
EXPECTED_LABELS_SHA256 = '719d36e9918b1fd09711c06d1b259fca08ca6eb4d973de1388d5e3ddecf22d35'
EXPECTED_RAW_DATA_SHA256 = '73bd0d3c17e75e7c0f10e4324ffa9ef81f6224cd2b3ba0e53e9f2aea7fde3212'
EXPECTED_PREFLIGHT_SIGNATURE = '961831943107801b06846ea7dfa69877601e0d2e0defd6e71c3a14c97ffa64cf'
EXPECTED_ROWS = 26527
EXPECTED_CLASS_COUNTS = {0: 9999, 1: 3961, 2: 12567}
CLASS_NAMES = ('Recyclable', 'Electronic', 'Organic')
CLASS_MAPPING = {0: 'Recyclable', 1: 'Electronic', 2: 'Organic'}
NUM_CLASSES = 3
N_FOLDS = 5
DEVELOPMENT_FOLD = 0
CONFIRMATION_FOLDS = (1, 2, 3, 4)
SEED = 42
HIDDEN_SIZE = 4096
LAST_FOUR_BLOCK_INDICES = (36, 37, 38, 39)
LAST_FOUR_BLOCK_NAMES = ('layer.36', 'layer.37', 'layer.38', 'layer.39')
EXPECTED_FEATURE_SHAPE = (EXPECTED_ROWS, 4, 2, HIDDEN_SIZE)
EXPECTED_FEATURE_DTYPE = torch.float16
CONTROL_NAME = 'regularized_mlp_final'
SPECIALIST_NAME = 'ro_specialist_final'
PRIMARY_CANDIDATE = 'ro_specialist_blend75'
ABLATION_CANDIDATE = 'ro_specialist_full'
FUSION_SPECS = {
    PRIMARY_CANDIDATE: {'specialist_weight': 0.75, 'role': 'primary'},
    ABLATION_CANDIDATE: {'specialist_weight': 1.0, 'role': 'full-replacement mechanism ablation'},
}
MAX_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 5
HEAD_LEARNING_RATE = 8e-4
HEAD_WEIGHT_DECAY = 1e-4
HEAD_BATCH_SIZE = 256
PREDICT_BATCH_SIZE = 512
GRAD_CLIP_MAX_NORM = 1.0
CONTROL_DROPOUT = 0.40
CONTROL_LABEL_SMOOTHING = 0.05
SPECIALIST_DROPOUT = 0.40
SPECIALIST_LABEL_SMOOTHING = 0.05
CONTROL_REPRODUCTION_TOLERANCE = 0.0002
EXPECTED_V7_FOLD0_CONTROL_MACRO_F1 = 0.9899180117906018
EXPECTED_V7_FOLD0_CONTROL_LOG_LOSS = 0.10112759881958534
EXPECTED_V7_FOLD0_CONTROL_TOTAL_ERRORS = 61
EXPECTED_V7_FOLD0_CONTROL_RO_ERRORS = 54
EXPECTED_V7_FOLD0_CONTROL_ELECTRONIC_F1 = 0.9955891619407687
EXPECTED_V7_FOLD0_LAST4_MACRO_F1 = 0.9880727960809867
EXPECTED_V7_CONFIRMATION_MACRO_F1 = 0.990190599402788
EXPECTED_V7_CONFIRMATION_TOTAL_ERRORS = 226
EXPECTED_V7_CONFIRMATION_RO_ERRORS = 187
EXPECTED_V7_CONFIRMATION_ELECTRONIC_F1 = 0.9938592347661785
FOLD0_GATES = {
    'minimum_macro_f1': 0.9903,
    'minimum_gain_over_reproduced_control': 0.0003,
    'minimum_ro_error_reduction': 3,
    'maximum_total_error_increase': 0,
    'maximum_electronic_f1_degradation': 0.0005,
    'minimum_corrected_samples': 4,
    'minimum_net_corrections': 3,
}
CONFIRMATION_GATES = {
    'minimum_aggregate_macro_f1': 0.9905,
    'minimum_folds_beating_reproduced_control': 3,
    'maximum_ro_errors_exclusive': EXPECTED_V7_CONFIRMATION_RO_ERRORS,
    'minimum_electronic_f1': 0.9933,
    'maximum_electronic_f1_degradation': 0.0005,
    'require_total_error_decline': True,
}
HIGH_CONFIDENCE_ERROR_THRESHOLD = 0.90
CONFIDENCE_BINS = (0.0, 0.50, 0.70, 0.80, 0.90, 0.95, 1.000001)
ARTIFACT_VALUE_TOLERANCE = 1e-12

KAGGLE_WORKING = Path('/kaggle/working')
if not KAGGLE_WORKING.is_dir():
    raise RuntimeError('V8 requires a Kaggle runtime with /kaggle/working.')
ARTIFACT_DIR = KAGGLE_WORKING / EXPERIMENT_NAME
HEADS_DIR = ARTIFACT_DIR / 'heads'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
HEADS_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
PIN_MEMORY = DEVICE.type == 'cuda'


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def sha256_json(value):
    payload = json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False)
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()


def json_default(value):
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    if torch.is_tensor(value):
        return value.detach().cpu().tolist()
    raise TypeError('Not JSON serializable: {}'.format(type(value).__name__))


def write_json_atomic(value, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        json.dump(value, handle, indent=2, sort_keys=True, default=json_default, allow_nan=False)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)


def write_csv_atomic(frame, path, index=False):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=index)
    os.replace(temporary, path)


def atomic_torch_save(value, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    if temporary.exists():
        temporary.unlink()
    with temporary.open('wb') as handle:
        torch.save(value, handle)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)


def read_json(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return json.load(handle)


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def fail_closed(stage, message, context=None):
    report = {
        'schema_version': V8_SCHEMA_VERSION,
        'revision_marker': V8_REVISION_MARKER,
        'pass': False,
        'failed_stage': stage,
        'error': str(message),
        'context': context or {},
        'action': 'Resolve the reported incompatibility and restart V8. No fallback, cache rebuild, test access, or submission is permitted.',
    }
    write_json_atomic(report, ARTIFACT_DIR / 'failure_report.json')
    raise RuntimeError('{}: {}'.format(stage, message))


seed_everything(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)
torch.set_float32_matmul_precision('high')

APPROACH_DECISION = {
    'schema_version': V8_SCHEMA_VERSION,
    'revision_marker': V8_REVISION_MARKER,
    'decision_timing': 'fixed before any V8 fold-0 scoring',
    'evidence_reviewed': [
        'V7 run_manifest.json, experiment_decision.json, head_selection.json',
        'V7 fold-0 comparison, aggregate metrics, all five head metrics/histories/reports/predictions',
        'V7 historical folds 1-4 metrics, reports, predictions, high-confidence errors, R/O gallery CSV and PNG',
        'V7 feature-cache manifest, fold/data signature, model/preprocess audit, preflight, runtime summary',
        'V7, V6 and V5 notebooks; competition technical guide; project handoff/context notes',
        'DINOv3 report, selected frozen-readout/layer-selection references, and waste-classification review',
    ],
    'verified_repository_evidence': {
        'v7_fold0_selected_macro_f1': EXPECTED_V7_FOLD0_CONTROL_MACRO_F1,
        'v7_fold0_selected_total_errors': EXPECTED_V7_FOLD0_CONTROL_TOTAL_ERRORS,
        'v7_fold0_selected_ro_errors': EXPECTED_V7_FOLD0_CONTROL_RO_ERRORS,
        'v7_historical_folds1_4_macro_f1': EXPECTED_V7_CONFIRMATION_MACRO_F1,
        'v7_historical_folds1_4_total_errors': EXPECTED_V7_CONFIRMATION_TOTAL_ERRORS,
        'v7_historical_folds1_4_ro_errors': EXPECTED_V7_CONFIRMATION_RO_ERRORS,
        'v7_historical_folds1_4_electronic_f1': EXPECTED_V7_CONFIRMATION_ELECTRONIC_F1,
    },
    'selected_family': 'binary Recyclable/Organic specialist with Electronic-preserving hierarchical probability-mass fusion',
    'primary_candidate': PRIMARY_CANDIDATE,
    'related_ablation': ABLATION_CANDIDATE,
    'expected_mechanism': 'learn the dominant R/O boundary directly while leaving the strong main-head Electronic probability and decision mask unchanged',
    'why_best_expected_value_now': 'R/O accounts for most verified residual errors, the final cached readout already beat the generic last-four alternative, and one binary probe is cheap while the fusion structurally protects Electronic behavior.',
    'rejected_alternatives': {
        'broad_classifier_redesign': 'V7 already screened five heads; the selected regularized final-layer MLP won clearly.',
        'generic_last_four_readout': 'The V7 last-four weighted-gated head underperformed the selected final-layer head.',
        'feature_space_augmentation': 'Higher design risk and broader search than justified by the deadline and concentrated error pattern.',
        'calibration_only': 'May improve probability quality but cannot by itself improve argmax Macro F1; threshold tuning would invite post-hoc overfit.',
        'mass_relabeling_or_sample_removal': 'The gallery shows ambiguity, not evidence for automatic label changes; official labels stay canonical.',
    },
    'key_risks': [
        'The binary specialist may overfit ambiguous or noisy R/O examples.',
        'A specialist trained without Electronic examples may emit arbitrary R/O probabilities on Electronic inputs; the fusion therefore preserves the main Electronic decision mask.',
        'Historical folds 1-4 informed the V8 direction and are confirmation folds, not pristine test-like assessment data.',
    ],
}

LOCKED_V8_CONTRACT = {
    'schema_version': V8_SCHEMA_VERSION,
    'revision_marker': V8_REVISION_MARKER,
    'model_id': MODEL_ID,
    'model_revision': EXPECTED_MODEL_REVISION,
    'v7_cache_signature': EXPECTED_CACHE_SIGNATURE,
    'fold_assignment_sha256': EXPECTED_FOLD_ASSIGNMENT_SHA256,
    'cache_only': True,
    'backbone_loaded_or_forwarded': False,
    'feature_readout': 'final cached block concat(CLS, mean true patches)',
    'control': {
        'name': CONTROL_NAME,
        'architecture': 'LayerNorm(8192)-Linear(512)-GELU-Dropout(0.40)-Linear(3)',
        'weighted_cross_entropy': True,
        'label_smoothing': CONTROL_LABEL_SMOOTHING,
    },
    'specialist': {
        'name': SPECIALIST_NAME,
        'training_labels': [0, 2],
        'architecture': 'LayerNorm(8192)-Linear(512)-GELU-Dropout(0.40)-Linear(2)',
        'weighted_cross_entropy': True,
        'label_smoothing': SPECIALIST_LABEL_SMOOTHING,
        'checkpoint_selection': 'binary validation Macro F1, then lower binary log loss',
    },
    'fusion_specs': FUSION_SPECS,
    'fusion_invariants': ['main Electronic probability preserved', 'main Electronic decision mask preserved', 'only R/O probability mass redistributed'],
    'training': {
        'seed': SEED,
        'max_epochs': MAX_EPOCHS,
        'patience': EARLY_STOPPING_PATIENCE,
        'optimizer': 'AdamW',
        'learning_rate': HEAD_LEARNING_RATE,
        'weight_decay': HEAD_WEIGHT_DECAY,
        'scheduler': 'CosineAnnealingLR',
        'batch_size': HEAD_BATCH_SIZE,
        'gradient_clip_max_norm': GRAD_CLIP_MAX_NORM,
    },
    'control_reproduction_tolerance': CONTROL_REPRODUCTION_TOLERANCE,
    'fold0_gates': FOLD0_GATES,
    'single_sample_dependency_guard': 'Require at least four corrected samples and at least three net corrections; high-confidence residuals are manual-review candidates only.',
    'confirmation_gates': CONFIRMATION_GATES,
    'candidate_selection': 'passing candidates ranked by Macro F1, then lower log loss, then fixed candidate order',
    'development_fold': DEVELOPMENT_FOLD,
    'confirmation_folds': list(CONFIRMATION_FOLDS),
    'official_test_accessed': False,
    'submission_created': False,
}
LOCKED_V8_CONTRACT_SIGNATURE = sha256_json(LOCKED_V8_CONTRACT)
write_json_atomic(LOCKED_V8_CONTRACT, ARTIFACT_DIR / 'locked_v8_contract.json')

assert tuple(FUSION_SPECS) == (PRIMARY_CANDIDATE, ABLATION_CANDIDATE)
assert MAX_EPOCHS == 15 and EARLY_STOPPING_PATIENCE == 5 and SEED == 42
assert CONTROL_DROPOUT == 0.40 and CONTROL_LABEL_SMOOTHING == 0.05
print('V8 artifacts:', ARTIFACT_DIR)
print('Head device:', DEVICE)
print('Locked V8 contract:', LOCKED_V8_CONTRACT_SIGNATURE)

V8 artifacts: /kaggle/working/v8_dinov3_vit7b_frozen_cache_upgrade
Head device: cuda:0
Locked V8 contract: 5241f8fc5f45ace336d07db424abd4cc59a2ec599219dc5182e21875b7fe97d6


## 4. V7 artifact/cache discovery

Discovery is bounded to `/kaggle/working/v7_dinov3_vit7b_frozen_raw`, each immediate Kaggle input dataset root, and one same-named child beneath each root. It checks only exact V7 marker paths and never recursively scans Kaggle storage. Exactly one complete V7 directory must resolve.

In [2]:
REQUIRED_V7_RELATIVE_PATHS = (
    'run_manifest.json',
    'experiment_decision.json',
    'head_selection.json',
    'fold0_head_comparison.csv',
    'fold0_head_metrics.json',
    'assessment_aggregate_comparison.csv',
    'assessment_per_fold_comparison.csv',
    'assessment_per_fold_metrics.json',
    'assessment_selected_metrics_folds1_4.json',
    'assessment_baseline_metrics_folds1_4.json',
    'assessment_selected_classification_report.csv',
    'assessment_baseline_classification_report.csv',
    'assessment_selected_predictions_folds1_4.csv',
    'assessment_baseline_predictions_folds1_4.csv',
    'selected_assessment_high_confidence_errors.csv',
    'selected_assessment_recyclable_organic_gallery.csv',
    'feature_cache_manifest.json',
    'data_and_fold_signature.json',
    'model_and_preprocess_audit.json',
    'preflight_report.json',
    'runtime_and_memory_summary.json',
    'fold_assignments.csv',
    'heads/development_regularized_mlp_final_fold0_seed42.pt',
)


def bounded_v7_candidates():
    roots = [KAGGLE_WORKING / V7_DIRECTORY_NAME, Path( "/kaggle/input/datasets/riantamaputra/" "v7-results/v7_dinov3_vit7b_frozen_raw"),]
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.is_dir():
        for dataset_root in sorted(path for path in kaggle_input.iterdir() if path.is_dir()):
            roots.extend((dataset_root, dataset_root / V7_DIRECTORY_NAME))
    unique_roots = []
    seen = set()
    for root in roots:
        key = str(root)
        if key not in seen:
            unique_roots.append(root)
            seen.add(key)
    return [
        root for root in unique_roots
        if all((root / relative).is_file() for relative in REQUIRED_V7_RELATIVE_PATHS)
        and (root / 'feature_cache' / 'shards').is_dir()
    ]


def iter_string_values(value):
    if isinstance(value, str):
        yield value
    elif isinstance(value, dict):
        for nested in value.values():
            yield from iter_string_values(nested)
    elif isinstance(value, (list, tuple)):
        for nested in value:
            yield from iter_string_values(nested)


def contains_forbidden_test_reference(value):
    for text in iter_string_values(value):
        tokens = [token.lower() for token in text.replace(chr(92), '/').split('/') if token]
        if any(token in {'test', 'template.csv', 'submission.csv'} for token in tokens):
            return True
    return False


resolved_candidates = bounded_v7_candidates()
if len(resolved_candidates) != 1:
    fail_closed(
        'v7_discovery',
        'Expected exactly one bounded complete V7 result directory, found {}.'.format(len(resolved_candidates)),
        {'candidates': [str(path) for path in resolved_candidates]},
    )
V7_ROOT = resolved_candidates[0]
if contains_forbidden_test_reference(str(V7_ROOT)):
    fail_closed('test_access_guard', 'Resolved V7 root contains a forbidden test/template/submission path component.')

V7_RUN_MANIFEST = read_json(V7_ROOT / 'run_manifest.json')
V7_EXPERIMENT_DECISION = read_json(V7_ROOT / 'experiment_decision.json')
V7_HEAD_SELECTION = read_json(V7_ROOT / 'head_selection.json')
V7_FOLD0_HEAD_METRICS = read_json(V7_ROOT / 'fold0_head_metrics.json')
V7_ASSESSMENT_PER_FOLD_METRICS = read_json(V7_ROOT / 'assessment_per_fold_metrics.json')
V7_SELECTED_ASSESSMENT_METRICS = read_json(V7_ROOT / 'assessment_selected_metrics_folds1_4.json')
V7_BASELINE_ASSESSMENT_METRICS = read_json(V7_ROOT / 'assessment_baseline_metrics_folds1_4.json')
V7_FEATURE_MANIFEST = read_json(V7_ROOT / 'feature_cache_manifest.json')
V7_DATA_SIGNATURE = read_json(V7_ROOT / 'data_and_fold_signature.json')
V7_MODEL_AUDIT = read_json(V7_ROOT / 'model_and_preprocess_audit.json')
V7_PREFLIGHT = read_json(V7_ROOT / 'preflight_report.json')
V7_RUNTIME_SUMMARY = read_json(V7_ROOT / 'runtime_and_memory_summary.json')
V7_FOLD0_COMPARISON = pd.read_csv(V7_ROOT / 'fold0_head_comparison.csv')
V7_ASSESSMENT_COMPARISON = pd.read_csv(V7_ROOT / 'assessment_aggregate_comparison.csv')
V7_ASSESSMENT_PER_FOLD_COMPARISON = pd.read_csv(V7_ROOT / 'assessment_per_fold_comparison.csv')
V7_SELECTED_ASSESSMENT_PREDICTIONS = pd.read_csv(V7_ROOT / 'assessment_selected_predictions_folds1_4.csv')
V7_BASELINE_ASSESSMENT_PREDICTIONS = pd.read_csv(V7_ROOT / 'assessment_baseline_predictions_folds1_4.csv')

for artifact_name, artifact_value in {
    'run_manifest': V7_RUN_MANIFEST,
    'experiment_decision': V7_EXPERIMENT_DECISION,
    'head_selection': V7_HEAD_SELECTION,
    'fold0_head_metrics': V7_FOLD0_HEAD_METRICS,
    'assessment_per_fold_metrics': V7_ASSESSMENT_PER_FOLD_METRICS,
    'selected_assessment_metrics': V7_SELECTED_ASSESSMENT_METRICS,
    'baseline_assessment_metrics': V7_BASELINE_ASSESSMENT_METRICS,
    'feature_manifest': V7_FEATURE_MANIFEST,
    'data_signature': V7_DATA_SIGNATURE,
    'model_audit': V7_MODEL_AUDIT,
    'preflight': V7_PREFLIGHT,
    'runtime_summary': V7_RUNTIME_SUMMARY,
}.items():
    if contains_forbidden_test_reference(artifact_value):
        fail_closed('test_access_guard', 'Forbidden test/template/submission path found in loaded V7 artifact {}.'.format(artifact_name))
for frame_name, frame in {
    'fold0_head_comparison': V7_FOLD0_COMPARISON,
    'assessment_aggregate_comparison': V7_ASSESSMENT_COMPARISON,
    'assessment_per_fold_comparison': V7_ASSESSMENT_PER_FOLD_COMPARISON,
    'selected_assessment_predictions': V7_SELECTED_ASSESSMENT_PREDICTIONS,
    'baseline_assessment_predictions': V7_BASELINE_ASSESSMENT_PREDICTIONS,
}.items():
    if contains_forbidden_test_reference(frame.to_dict(orient='list')):
        fail_closed('test_access_guard', 'Forbidden test/template/submission reference found in loaded V7 table {}.'.format(frame_name))
for frame_name, frame in {
    'selected_assessment_predictions': V7_SELECTED_ASSESSMENT_PREDICTIONS,
    'baseline_assessment_predictions': V7_BASELINE_ASSESSMENT_PREDICTIONS,
}.items():
    require('image_id' in frame.columns, '{} lacks image_id.'.format(frame_name))
    if frame['image_id'].astype(str).map(contains_forbidden_test_reference).any():
        fail_closed('test_access_guard', 'Forbidden path found in {}.'.format(frame_name))

selected_fold0_matches = [row for row in V7_FOLD0_HEAD_METRICS if row.get('candidate') == CONTROL_NAME]
require(len(selected_fold0_matches) == 1, 'V7 fold-0 selected-control metrics are absent or ambiguous.')
V7_FOLD0_CONTROL_METRICS = selected_fold0_matches[0]
last4_matches = [row for row in V7_FOLD0_HEAD_METRICS if row.get('candidate') == 'last4_layer_weighted_gated']
require(len(last4_matches) == 1, 'V7 last-four-layer comparator metrics are absent or ambiguous.')
V7_LAST4_METRICS = last4_matches[0]
v7_fold0_control_ro_errors = V7_FOLD0_CONTROL_METRICS['recyclable_to_organic'] + V7_FOLD0_CONTROL_METRICS['organic_to_recyclable']
require(V7_HEAD_SELECTION.get('selected_head') == CONTROL_NAME, 'V7 selected head is not regularized_mlp_final.')
require(V7_RUN_MANIFEST.get('feature_cache_signature') == EXPECTED_CACHE_SIGNATURE, 'V7 run manifest cache signature differs from the reviewed cache.')
require(V7_RUN_MANIFEST.get('selected_head') == CONTROL_NAME, 'V7 run manifest selected head differs from the reviewed control.')
require(V7_RUN_MANIFEST.get('competition_compliance', {}).get('competition_test_accessed') is False, 'V7 manifest does not attest zero test access.')
require(V7_RUN_MANIFEST.get('competition_compliance', {}).get('submission_created') is False, 'V7 manifest reports a submission.')

for observed, expected, label in (
    (V7_FOLD0_CONTROL_METRICS['macro_f1'], EXPECTED_V7_FOLD0_CONTROL_MACRO_F1, 'fold0 control Macro F1'),
    (V7_FOLD0_CONTROL_METRICS['multiclass_log_loss'], EXPECTED_V7_FOLD0_CONTROL_LOG_LOSS, 'fold0 control log loss'),
    (V7_FOLD0_CONTROL_METRICS['classification_report']['Electronic']['f1-score'], EXPECTED_V7_FOLD0_CONTROL_ELECTRONIC_F1, 'fold0 control Electronic F1'),
    (V7_LAST4_METRICS['macro_f1'], EXPECTED_V7_FOLD0_LAST4_MACRO_F1, 'fold0 last-four comparator Macro F1'),
    (V7_SELECTED_ASSESSMENT_METRICS['macro_f1'], EXPECTED_V7_CONFIRMATION_MACRO_F1, 'historical folds1-4 Macro F1'),
    (V7_SELECTED_ASSESSMENT_METRICS['classification_report']['Electronic']['f1-score'], EXPECTED_V7_CONFIRMATION_ELECTRONIC_F1, 'historical Electronic F1'),
):
    require(math.isclose(float(observed), float(expected), rel_tol=0.0, abs_tol=ARTIFACT_VALUE_TOLERANCE), '{} disagrees with the reviewed bundle.'.format(label))
require(V7_FOLD0_CONTROL_METRICS['total_errors'] == EXPECTED_V7_FOLD0_CONTROL_TOTAL_ERRORS, 'V7 fold-0 control error count differs from reviewed evidence.')
require(v7_fold0_control_ro_errors == EXPECTED_V7_FOLD0_CONTROL_RO_ERRORS, 'V7 fold-0 control R/O error count differs from reviewed evidence.')
require(V7_LAST4_METRICS['macro_f1'] < V7_FOLD0_CONTROL_METRICS['macro_f1'], 'V7 last-four comparator no longer underperforms the selected final-layer control.')

selected_assessment = V7_SELECTED_ASSESSMENT_PREDICTIONS.sort_values('row_index').reset_index(drop=True)
baseline_assessment = V7_BASELINE_ASSESSMENT_PREDICTIONS.sort_values('row_index').reset_index(drop=True)
require(len(selected_assessment) == len(baseline_assessment) == 21221, 'V7 assessment prediction row counts are invalid.')
require(selected_assessment['row_index'].is_unique and baseline_assessment['row_index'].is_unique, 'V7 assessment prediction indices are duplicated.')
require(selected_assessment['row_index'].tolist() == baseline_assessment['row_index'].tolist(), 'V7 assessment prediction indices are misaligned.')
require(selected_assessment['label'].astype(int).tolist() == baseline_assessment['label'].astype(int).tolist(), 'V7 assessment prediction labels are misaligned.')
selected_labels = selected_assessment['label'].to_numpy(dtype=np.int64)
selected_predictions = selected_assessment['prediction'].to_numpy(dtype=np.int64)
baseline_predictions = baseline_assessment['prediction'].to_numpy(dtype=np.int64)
selected_errors = selected_predictions != selected_labels
baseline_errors = baseline_predictions != selected_labels
selected_ro_errors = int((((selected_labels == 0) & (selected_predictions == 2)) | ((selected_labels == 2) & (selected_predictions == 0))).sum())
selected_total_errors = int(selected_errors.sum())
require(selected_total_errors == EXPECTED_V7_CONFIRMATION_TOTAL_ERRORS, 'V7 selected assessment error count differs from reviewed evidence.')
require(selected_ro_errors == EXPECTED_V7_CONFIRMATION_RO_ERRORS, 'V7 selected assessment R/O error count differs from reviewed evidence.')

with torch.no_grad():
    v7_control_checkpoint = torch.load(
        V7_ROOT / 'heads' / 'development_regularized_mlp_final_fold0_seed42.pt',
        map_location='cpu',
        weights_only=False,
    )
v7_control_checkpoint_metadata = v7_control_checkpoint.get('metadata')
v7_control_state = v7_control_checkpoint.get('state_dict')
require(isinstance(v7_control_checkpoint_metadata, dict) and isinstance(v7_control_state, dict), 'V7 control checkpoint payload is invalid.')
if contains_forbidden_test_reference(v7_control_checkpoint_metadata):
    fail_closed('test_access_guard', 'Forbidden test/template/submission reference found in V7 control checkpoint metadata.')
expected_control_metadata = {
    'candidate': CONTROL_NAME,
    'seed': SEED,
    'cache_signature': EXPECTED_CACHE_SIGNATURE,
    'max_epochs': MAX_EPOCHS,
    'patience': EARLY_STOPPING_PATIENCE,
    'learning_rate': HEAD_LEARNING_RATE,
    'weight_decay': HEAD_WEIGHT_DECAY,
    'gradient_clip_max_norm': GRAD_CLIP_MAX_NORM,
}
for key, expected in expected_control_metadata.items():
    require(v7_control_checkpoint_metadata.get(key) == expected, 'V7 control checkpoint metadata mismatch for {}.'.format(key))
require(v7_control_checkpoint_metadata.get('candidate_spec', {}).get('dropout') == CONTROL_DROPOUT, 'V7 control dropout differs from the locked reproduction.')
require(v7_control_checkpoint_metadata.get('candidate_spec', {}).get('label_smoothing') == CONTROL_LABEL_SMOOTHING, 'V7 control label smoothing differs from the locked reproduction.')
expected_control_state_shapes = {
    'network.0.weight': (2 * HIDDEN_SIZE,),
    'network.0.bias': (2 * HIDDEN_SIZE,),
    'network.1.weight': (512, 2 * HIDDEN_SIZE),
    'network.1.bias': (512,),
    'network.4.weight': (NUM_CLASSES, 512),
    'network.4.bias': (NUM_CLASSES,),
}
require(set(v7_control_state) == set(expected_control_state_shapes), 'V7 control checkpoint state keys differ from the expected architecture.')
for key, expected_shape in expected_control_state_shapes.items():
    require(tuple(v7_control_state[key].shape) == expected_shape, 'V7 control checkpoint tensor shape mismatch for {}.'.format(key))
del v7_control_checkpoint, v7_control_state

V7_ARTIFACT_AUDIT = {
    'schema_version': V8_SCHEMA_VERSION,
    'revision_marker': V8_REVISION_MARKER,
    'pass': True,
    'v7_root': str(V7_ROOT),
    'required_artifact_count': len(REQUIRED_V7_RELATIVE_PATHS),
    'model_id': V7_RUN_MANIFEST['model_and_preprocess']['model_id'],
    'model_revision': V7_RUN_MANIFEST['model_and_preprocess']['resolved_revision'],
    'cache_signature': V7_RUN_MANIFEST['feature_cache_signature'],
    'fold0': {
        'selected_control_macro_f1': V7_FOLD0_CONTROL_METRICS['macro_f1'],
        'selected_control_log_loss': V7_FOLD0_CONTROL_METRICS['multiclass_log_loss'],
        'selected_control_total_errors': V7_FOLD0_CONTROL_METRICS['total_errors'],
        'selected_control_ro_errors': V7_FOLD0_CONTROL_METRICS['recyclable_to_organic'] + V7_FOLD0_CONTROL_METRICS['organic_to_recyclable'],
        'selected_control_electronic_f1': V7_FOLD0_CONTROL_METRICS['classification_report']['Electronic']['f1-score'],
        'last4_weighted_gated_macro_f1': V7_LAST4_METRICS['macro_f1'],
    },
    'historical_folds1_4': {
        'selected_macro_f1': V7_SELECTED_ASSESSMENT_METRICS['macro_f1'],
        'selected_log_loss': V7_SELECTED_ASSESSMENT_METRICS['multiclass_log_loss'],
        'baseline_log_loss': V7_BASELINE_ASSESSMENT_METRICS['multiclass_log_loss'],
        'selected_total_errors': selected_total_errors,
        'selected_ro_errors': selected_ro_errors,
        'ro_share_of_selected_errors': selected_ro_errors / selected_total_errors,
        'baseline_errors_fixed_by_selected': int((baseline_errors & ~selected_errors).sum()),
        'newly_broken_by_selected': int((~baseline_errors & selected_errors).sum()),
        'selected_electronic_f1': V7_SELECTED_ASSESSMENT_METRICS['classification_report']['Electronic']['f1-score'],
    },
    'control_recipe_resolved_from_checkpoint_metadata': v7_control_checkpoint_metadata,
    'test_access_guard_passed': True,
}
write_json_atomic(V7_ARTIFACT_AUDIT, ARTIFACT_DIR / 'v7_artifact_audit.json')
APPROACH_DECISION['runtime_verified_v7_evidence'] = V7_ARTIFACT_AUDIT
write_json_atomic(APPROACH_DECISION, ARTIFACT_DIR / 'approach_decision.json')
print('Resolved V7 root:', V7_ROOT)
print('Verified historical R/O error share: {:.3%}'.format(V7_ARTIFACT_AUDIT['historical_folds1_4']['ro_share_of_selected_errors']))

Resolved V7 root: /kaggle/input/datasets/riantamaputra/v7-results/v7_dinov3_vit7b_frozen_raw
Verified historical R/O error share: 82.743%


## 5. Cache compatibility audit

The V7 cache must be schema 2 with the official pretrained final LayerNorm applied per token before CLS/true-patch pooling. Manifest ranges, shard files, metadata, labels, folds, image IDs, dtypes, finiteness, and all identity hashes are checked. V8 retains only the final cached block in memory after validating every four-block shard.

In [3]:
CACHE_COMPATIBILITY_AUDIT = {
    'schema_version': V8_SCHEMA_VERSION,
    'revision_marker': V8_REVISION_MARKER,
    'pass': False,
    'v7_root': str(V7_ROOT),
    'cache_signature': V7_FEATURE_MANIFEST.get('cache_signature'),
}
try:
    cache_identity = V7_FEATURE_MANIFEST.get('cache_identity', {})
    require(V7_FEATURE_MANIFEST.get('schema_version') == 2, 'V7 feature-cache schema must be 2.')
    require(V7_FEATURE_MANIFEST.get('revision_marker') == V7_REVISION_MARKER, 'V7 cache revision marker is incompatible.')
    require(V7_FEATURE_MANIFEST.get('complete') is True, 'V7 cache manifest is incomplete.')
    require(V7_FEATURE_MANIFEST.get('cache_signature') == EXPECTED_CACHE_SIGNATURE, 'V7 cache signature differs from the reviewed cache.')
    require(V7_FEATURE_MANIFEST.get('rows') == EXPECTED_ROWS, 'V7 cache row count is invalid.')
    require(tuple(V7_FEATURE_MANIFEST.get('feature_tensor_shape', ())) == EXPECTED_FEATURE_SHAPE, 'V7 cache tensor shape is invalid.')
    require(cache_identity.get('schema_version') == 2, 'V7 cache identity schema is incompatible.')
    require(cache_identity.get('revision_marker') == V7_REVISION_MARKER, 'V7 cache identity revision is incompatible.')
    require(cache_identity.get('model_id') == MODEL_ID and 'lvd1689m' in cache_identity.get('model_id', '').lower(), 'V7 cache is not the locked LVD-1689M ViT-7B model.')
    require('sat493m' not in cache_identity.get('model_id', '').lower(), 'SAT-493M cache is forbidden.')
    require(cache_identity.get('model_revision') == EXPECTED_MODEL_REVISION, 'V7 immutable model revision differs.')
    require(cache_identity.get('split') == 'official_raw_train', 'V7 cache is not the official labeled training split.')
    require(cache_identity.get('image_size') == 256, 'V7 cache image size is not 256.')
    require(cache_identity.get('feature_view') == 'orig', 'V7 cache is not original-view only.')
    require(cache_identity.get('backbone_dtype') == 'float16', 'V7 backbone precision is incompatible.')
    require(cache_identity.get('feature_storage_dtype') == 'float16', 'V7 feature storage precision is incompatible.')
    require(cache_identity.get('hidden_size') == HIDDEN_SIZE, 'V7 hidden size is not 4096.')
    require(tuple(cache_identity.get('last_four_block_indices', ())) == LAST_FOUR_BLOCK_INDICES, 'V7 cached block indices differ.')
    require(tuple(cache_identity.get('last_four_block_names', ())) == LAST_FOUR_BLOCK_NAMES, 'V7 cached block names differ.')
    require(cache_identity.get('pooled_layout') == ['cls', 'mean_true_patch_tokens'], 'V7 cache does not contain exactly CLS and true-patch mean.')
    require(cache_identity.get('per_token_normalization') == 'official pretrained DINOv3 final LayerNorm', 'V7 cache lacks the official final LayerNorm contract.')
    require(cache_identity.get('normalization_timing') == 'before CLS extraction and true-patch pooling', 'V7 cache normalization timing is incompatible.')
    require(cache_identity.get('final_block_source') == 'model_output.last_hidden_state', 'V7 final cached block is not official last_hidden_state.')
    require(cache_identity.get('fold_assignment_sha256') == EXPECTED_FOLD_ASSIGNMENT_SHA256, 'V7 cache fold signature differs.')
    require(cache_identity.get('ordered_image_ids_sha256') == EXPECTED_ORDERED_IMAGE_IDS_SHA256, 'V7 cache image ordering differs.')
    require(cache_identity.get('labels_sha256') == EXPECTED_LABELS_SHA256, 'V7 cache label signature differs.')
    require(cache_identity.get('raw_data_sha256') == EXPECTED_RAW_DATA_SHA256, 'V7 raw-data signature differs.')
    require(cache_identity.get('preflight_contract_signature') == EXPECTED_PREFLIGHT_SIGNATURE, 'V7 cache preflight signature differs.')
    require(V7_MODEL_AUDIT.get('model_id') == MODEL_ID, 'V7 model audit model ID differs.')
    require(V7_MODEL_AUDIT.get('resolved_revision') == EXPECTED_MODEL_REVISION, 'V7 model audit revision differs.')
    require(V7_MODEL_AUDIT.get('trainable_parameter_count') == 0, 'V7 backbone was not fully frozen.')
    require(V7_MODEL_AUDIT.get('hidden_size') == HIDDEN_SIZE, 'V7 model audit hidden size differs.')
    require(V7_MODEL_AUDIT.get('final_norm_is_frozen') is True, 'V7 official final norm was not frozen.')
    require(V7_MODEL_AUDIT.get('cached_representation') == 'official pretrained final LayerNorm per token before CLS/true-patch pooling', 'V7 model audit representation differs.')
    require(V7_PREFLIGHT.get('pass') is True and V7_PREFLIGHT.get('schema_version') == 2, 'V7 strict preflight did not pass schema 2.')
    require(V7_PREFLIGHT.get('preflight_contract_signature') == EXPECTED_PREFLIGHT_SIGNATURE, 'V7 preflight signature differs.')
    final_norm_equivalence = V7_PREFLIGHT['stages']['B_sharded_token_memory_audit']['official_final_norm_equivalence']
    require(isinstance(final_norm_equivalence, list) and len(final_norm_equivalence) == 2, 'V7 final-norm equivalence audit is absent or incomplete.')
    require(all(item.get('pass') is True for item in final_norm_equivalence), 'V7 final-norm equivalence audit failed.')

    fold_assignments = pd.read_csv(V7_ROOT / 'fold_assignments.csv')
    require(list(fold_assignments.columns) == ['image_id', 'label', 'class_name', 'fold'], 'V7 fold assignment columns differ.')
    require(len(fold_assignments) == EXPECTED_ROWS, 'V7 fold assignments have the wrong row count.')
    require(fold_assignments['image_id'].is_unique, 'V7 fold assignments contain duplicate image IDs.')
    require(fold_assignments['image_id'].astype(str).tolist() == sorted(fold_assignments['image_id'].astype(str).tolist()), 'V7 fold assignments are not in canonical image-ID order.')
    require(set(fold_assignments['label'].astype(int)) == set(CLASS_MAPPING), 'V7 fold labels are invalid.')
    require(set(fold_assignments['fold'].astype(int)) == set(range(N_FOLDS)), 'V7 fold domain is invalid.')
    expected_class_names = fold_assignments['label'].astype(int).map(CLASS_MAPPING)
    require(expected_class_names.tolist() == fold_assignments['class_name'].astype(str).tolist(), 'V7 class names do not match labels.')
    require(not fold_assignments['image_id'].astype(str).map(contains_forbidden_test_reference).any(), 'V7 fold assignments contain a forbidden path.')
    observed_class_counts = {int(key): int(value) for key, value in fold_assignments['label'].value_counts().sort_index().items()}
    require(observed_class_counts == EXPECTED_CLASS_COUNTS, 'V7 official class counts differ.')
    ordered_image_ids = fold_assignments['image_id'].astype(str).tolist()
    ordered_labels = fold_assignments['label'].astype(int).tolist()
    fold_records = [
        {'image_id': str(row.image_id), 'label': int(row.label), 'fold': int(row.fold)}
        for row in fold_assignments.itertuples(index=False)
    ]
    require(sha256_json(ordered_image_ids) == EXPECTED_ORDERED_IMAGE_IDS_SHA256, 'Recomputed ordered image-ID signature differs.')
    require(sha256_json(ordered_labels) == EXPECTED_LABELS_SHA256, 'Recomputed label signature differs.')
    require(sha256_json(fold_records) == EXPECTED_FOLD_ASSIGNMENT_SHA256, 'Recomputed fold signature differs.')
    require(V7_DATA_SIGNATURE.get('fold_assignment_sha256') == EXPECTED_FOLD_ASSIGNMENT_SHA256, 'V7 data-and-fold signature differs.')
    require(V7_DATA_SIGNATURE.get('raw_data_sha256') == EXPECTED_RAW_DATA_SHA256, 'V7 data audit raw hash differs.')
    require(V7_DATA_SIGNATURE.get('raw_class_counts') == {str(key): value for key, value in EXPECTED_CLASS_COUNTS.items()}, 'V7 data audit class counts differ.')
    require(V7_DATA_SIGNATURE.get('official_class_codes') == {str(key): value for key, value in CLASS_MAPPING.items()}, 'V7 label mapping differs.')

    manifest_shards = sorted(V7_FEATURE_MANIFEST.get('shards', []), key=lambda record: int(record['start']))
    require(len(manifest_shards) == V7_FEATURE_MANIFEST.get('shard_count') == 104, 'V7 shard count is invalid.')
    expected_start = 0
    expected_shard_names = []
    for shard_index, record in enumerate(manifest_shards):
        start, end = int(record['start']), int(record['end'])
        require(record.get('shard_index') == shard_index, 'V7 shard indices are not canonical.')
        require(start == expected_start and end > start, 'V7 shard ranges overlap or contain a gap.')
        require(int(record.get('rows')) == end - start, 'V7 shard row metadata is invalid.')
        expected_shard_names.append('train_{:05d}_{:05d}.pt'.format(start, end))
        expected_start = end
    require(expected_start == EXPECTED_ROWS, 'V7 shard ranges do not cover all rows.')
    shards_dir = V7_ROOT / 'feature_cache' / 'shards'
    actual_shard_names = sorted(path.name for path in shards_dir.glob('*.pt') if path.is_file())
    require(actual_shard_names == sorted(expected_shard_names), 'V7 shard files are missing, duplicated, or unexpected.')

    cached_final_features = torch.empty((EXPECTED_ROWS, 2, HIDDEN_SIZE), dtype=EXPECTED_FEATURE_DTYPE)
    cached_labels = torch.empty(EXPECTED_ROWS, dtype=torch.long)
    cached_folds = torch.empty(EXPECTED_ROWS, dtype=torch.long)
    for record, shard_name in zip(manifest_shards, expected_shard_names):
        start, end = int(record['start']), int(record['end'])
        expected_frame = fold_assignments.iloc[start:end]
        expected_metadata = {
            'schema_version': 2,
            'revision_marker': V7_REVISION_MARKER,
            'cache_signature': EXPECTED_CACHE_SIGNATURE,
            'start': start,
            'end': end,
            'rows': end - start,
            'image_ids_sha256': sha256_json(expected_frame['image_id'].astype(str).tolist()),
            'labels_sha256': sha256_json(expected_frame['label'].astype(int).tolist()),
            'folds_sha256': sha256_json(expected_frame['fold'].astype(int).tolist()),
            'feature_shape': [end - start, 4, 2, HIDDEN_SIZE],
            'feature_dtype': 'float16',
        }
        payload = torch.load(shards_dir / shard_name, map_location='cpu', weights_only=False)
        require(isinstance(payload, dict) and payload.get('metadata') == expected_metadata, 'V7 shard metadata mismatch: {}.'.format(shard_name))
        features = payload.get('features')
        labels = payload.get('labels')
        folds = payload.get('folds')
        image_ids = payload.get('image_ids')
        require(torch.is_tensor(features) and tuple(features.shape) == tuple(expected_metadata['feature_shape']), 'V7 shard feature shape mismatch: {}.'.format(shard_name))
        require(features.dtype == EXPECTED_FEATURE_DTYPE and torch.isfinite(features).all().item(), 'V7 shard feature dtype/finiteness failed: {}.'.format(shard_name))
        require(torch.is_tensor(labels) and labels.dtype == torch.long, 'V7 shard labels are invalid: {}.'.format(shard_name))
        require(torch.is_tensor(folds) and folds.dtype == torch.long, 'V7 shard folds are invalid: {}.'.format(shard_name))
        require(torch.equal(labels, torch.tensor(expected_frame['label'].to_numpy(dtype=np.int64), dtype=torch.long)), 'V7 shard labels are misaligned: {}.'.format(shard_name))
        require(torch.equal(folds, torch.tensor(expected_frame['fold'].to_numpy(dtype=np.int64), dtype=torch.long)), 'V7 shard folds are misaligned: {}.'.format(shard_name))
        require(image_ids == expected_frame['image_id'].astype(str).tolist(), 'V7 shard image IDs are misaligned: {}.'.format(shard_name))
        cached_final_features[start:end].copy_(features[:, -1])
        cached_labels[start:end].copy_(labels)
        cached_folds[start:end].copy_(folds)
        del payload, features, labels, folds, image_ids
    require(torch.isfinite(cached_final_features).all().item(), 'Assembled final-block cache contains NaN/Inf.')
    require(torch.equal(cached_labels, torch.tensor(fold_assignments['label'].to_numpy(dtype=np.int64), dtype=torch.long)), 'Assembled cache labels are misaligned.')
    require(torch.equal(cached_folds, torch.tensor(fold_assignments['fold'].to_numpy(dtype=np.int64), dtype=torch.long)), 'Assembled cache folds are misaligned.')

    CACHE_SIGNATURE = EXPECTED_CACHE_SIGNATURE
    MODEL_REVISION = EXPECTED_MODEL_REVISION
    FOLD_SIGNATURE = EXPECTED_FOLD_ASSIGNMENT_SHA256
    CACHE_COMPATIBILITY_AUDIT.update({
        'pass': True,
        'model_id': MODEL_ID,
        'model_revision': MODEL_REVISION,
        'cache_schema_version': 2,
        'cache_revision_marker': V7_REVISION_MARKER,
        'fold_assignment_sha256': FOLD_SIGNATURE,
        'rows': EXPECTED_ROWS,
        'official_class_counts': EXPECTED_CLASS_COUNTS,
        'feature_shape_validated_per_shard': list(EXPECTED_FEATURE_SHAPE),
        'retained_final_feature_shape': list(cached_final_features.shape),
        'feature_dtype': str(cached_final_features.dtype),
        'shard_count': len(manifest_shards),
        'contiguous_non_overlapping_coverage': True,
        'duplicate_or_missing_sample_indices': False,
        'official_final_layernorm_before_pooling': True,
        'backbone_frozen': True,
        'test_access_guard_passed': True,
    })
    write_json_atomic(CACHE_COMPATIBILITY_AUDIT, ARTIFACT_DIR / 'cache_compatibility_audit.json')
except Exception as exc:
    CACHE_COMPATIBILITY_AUDIT.update({'pass': False, 'error_type': type(exc).__name__, 'error': str(exc)})
    write_json_atomic(CACHE_COMPATIBILITY_AUDIT, ARTIFACT_DIR / 'cache_compatibility_audit.json')
    fail_closed('cache_compatibility_audit', str(exc), CACHE_COMPATIBILITY_AUDIT)

print('Validated V7 cache:', tuple(cached_final_features.shape), cached_final_features.dtype)
print('Validated shards:', CACHE_COMPATIBILITY_AUDIT['shard_count'])

Validated V7 cache: (26527, 2, 4096) torch.float16
Validated shards: 104


## 6. Fold and label validation

Image IDs are retained only for alignment and human-readable diagnostics. Models receive only cached visual tensors and official labels. Fold 0 remains the sole development fold; folds 1–4 are historical assessment folds reused conditionally for confirmation, not pristine test-like data.

In [4]:
fold_label_rows = []
for fold in range(N_FOLDS):
    fold_mask = cached_folds.numpy() == fold
    counts = np.bincount(cached_labels.numpy()[fold_mask], minlength=NUM_CLASSES)
    require((counts > 0).all(), 'Fold {} is missing a class.'.format(fold))
    fold_label_rows.append({
        'fold': fold,
        'rows': int(fold_mask.sum()),
        **{'class_{}_rows'.format(label): int(counts[label]) for label in range(NUM_CLASSES)},
    })
FOLD_LABEL_AUDIT = {
    'schema_version': V8_SCHEMA_VERSION,
    'revision_marker': V8_REVISION_MARKER,
    'pass': True,
    'fold_assignment_sha256': FOLD_SIGNATURE,
    'development_fold': DEVELOPMENT_FOLD,
    'historical_assessment_folds_reused_for_confirmation': list(CONFIRMATION_FOLDS),
    'official_test_is_not_a_fold': True,
    'predictive_inputs': ['cached final-layer CLS vector', 'cached final-layer true-patch mean vector'],
    'non_predictive_alignment_fields': ['row_index', 'image_id', 'fold'],
    'fold_rows': fold_label_rows,
}
write_json_atomic(FOLD_LABEL_AUDIT, ARTIFACT_DIR / 'fold_label_audit.json')
print(pd.DataFrame(fold_label_rows))

   fold  rows  class_0_rows  class_1_rows  class_2_rows
0     0  5306          2000           792          2514
1     1  5306          2000           792          2514
2     2  5305          2000           792          2513
3     3  5305          2000           792          2513
4     4  5305          1999           793          2513


## 7. Fixed candidate definitions established before scoring

The control exactly reproduces V7 `regularized_mlp_final`: LayerNorm, 512-unit GELU MLP, dropout 0.40, weighted cross-entropy with label smoothing 0.05, AdamW, cosine decay, 15 epochs, patience 5, and seed 42. The specialist uses the same recipe with two outputs and trains only on labels 0 and 2. Its checkpoint is selected by binary validation Macro F1. No candidate, weight, threshold, or gate changes after fold-0 results.

In [5]:
class CachedFinalRows(Dataset):
    def __init__(self, indices):
        self.indices = [int(index) for index in indices]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        index = self.indices[position]
        return cached_final_features[index], cached_labels[index], index


class V7RegularizedMLPFinal(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.LayerNorm(2 * HIDDEN_SIZE),
            nn.Linear(2 * HIDDEN_SIZE, 512),
            nn.GELU(),
            nn.Dropout(CONTROL_DROPOUT),
            nn.Linear(512, NUM_CLASSES),
        )

    def forward(self, features):
        return self.network(torch.cat((features[:, 0], features[:, 1]), dim=1))


class ROSpecialistFinal(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.LayerNorm(2 * HIDDEN_SIZE),
            nn.Linear(2 * HIDDEN_SIZE, 512),
            nn.GELU(),
            nn.Dropout(SPECIALIST_DROPOUT),
            nn.Linear(512, 2),
        )

    def forward(self, features):
        return self.network(torch.cat((features[:, 0], features[:, 1]), dim=1))


def normalize_probabilities_from_logits(logits):
    logits = np.asarray(logits, dtype=np.float64)
    require(logits.ndim == 2 and np.isfinite(logits).all(), 'Logits are incomplete or non-finite.')
    shifted = logits - logits.max(axis=1, keepdims=True)
    probabilities = np.exp(shifted)
    require(np.isfinite(probabilities).all(), 'Exponentiated logits are non-finite.')
    row_sums = probabilities.sum(axis=1, keepdims=True)
    require((row_sums > 0).all() and np.isfinite(row_sums).all(), 'Probability row sums are invalid.')
    probabilities = probabilities.astype(np.float64)
    probabilities /= probabilities.sum(axis=1, keepdims=True)
    require(np.isfinite(probabilities).all(), 'Probabilities contain NaN/Inf.')
    require(np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-12, rtol=0.0), 'Probabilities do not sum to one.')
    return probabilities


def normalize_probability_matrix(probabilities):
    probabilities = np.asarray(probabilities, dtype=np.float64).copy()
    require(probabilities.ndim == 2 and np.isfinite(probabilities).all() and (probabilities >= 0).all(), 'Probability matrix is invalid.')
    row_sums = probabilities.sum(axis=1, keepdims=True)
    require((row_sums > 0).all(), 'Probability matrix has a non-positive row sum.')
    probabilities = probabilities.astype(np.float64)
    probabilities /= probabilities.sum(axis=1, keepdims=True)
    require(np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-12, rtol=0.0), 'Renormalized rows do not sum to one.')
    return probabilities


def confidence_summary(values):
    values = np.asarray(values, dtype=np.float64)
    if not len(values):
        return {'count': 0, 'mean': None, 'std': None, 'q10': None, 'median': None, 'q90': None}
    return {
        'count': len(values),
        'mean': float(values.mean()),
        'std': float(values.std()),
        'q10': float(np.quantile(values, 0.10)),
        'median': float(np.quantile(values, 0.50)),
        'q90': float(np.quantile(values, 0.90)),
    }


def metric_bundle(labels, probabilities):
    labels = np.asarray(labels, dtype=np.int64)
    probabilities = normalize_probability_matrix(probabilities)
    require(probabilities.shape == (len(labels), NUM_CLASSES), 'Three-class metric probability shape is invalid.')
    predictions = probabilities.argmax(axis=1)
    matrix = confusion_matrix(labels, predictions, labels=list(range(NUM_CLASSES)))
    top_two = np.sort(probabilities, axis=1)[:, -2:]
    confidence = top_two[:, 1]
    margin = top_two[:, 1] - top_two[:, 0]
    correct = predictions == labels
    report = classification_report(
        labels,
        predictions,
        labels=list(range(NUM_CLASSES)),
        target_names=list(CLASS_NAMES),
        output_dict=True,
        zero_division=0,
    )
    return {
        'rows': len(labels),
        'macro_f1': float(f1_score(labels, predictions, labels=list(range(NUM_CLASSES)), average='macro')),
        'accuracy': float(accuracy_score(labels, predictions)),
        'multiclass_log_loss': float(log_loss(labels, probabilities, labels=list(range(NUM_CLASSES)))),
        'classification_report': report,
        'confusion_matrix': matrix.tolist(),
        'recyclable_to_organic': int(matrix[0, 2]),
        'organic_to_recyclable': int(matrix[2, 0]),
        'total_errors': int((~correct).sum()),
        'confidence_summary': {
            'all': confidence_summary(confidence),
            'correct': confidence_summary(confidence[correct]),
            'errors': confidence_summary(confidence[~correct]),
        },
        'top_two_margin_summary': {
            'all': confidence_summary(margin),
            'correct': confidence_summary(margin[correct]),
            'errors': confidence_summary(margin[~correct]),
        },
    }


def binary_metric_bundle(binary_labels, probabilities):
    binary_labels = np.asarray(binary_labels, dtype=np.int64)
    probabilities = normalize_probability_matrix(probabilities)
    require(probabilities.shape == (len(binary_labels), 2), 'Binary metric probability shape is invalid.')
    predictions = probabilities.argmax(axis=1)
    report = classification_report(
        binary_labels,
        predictions,
        labels=[0, 1],
        target_names=['Recyclable', 'Organic'],
        output_dict=True,
        zero_division=0,
    )
    return {
        'rows': len(binary_labels),
        'macro_f1': float(f1_score(binary_labels, predictions, labels=[0, 1], average='macro')),
        'accuracy': float(accuracy_score(binary_labels, predictions)),
        'binary_log_loss': float(log_loss(binary_labels, probabilities, labels=[0, 1])),
        'classification_report': report,
        'confusion_matrix': confusion_matrix(binary_labels, predictions, labels=[0, 1]).tolist(),
        'total_errors': int((predictions != binary_labels).sum()),
    }


def report_frame(report, label_names):
    rows = []
    for label_name in list(label_names) + ['macro avg', 'weighted avg']:
        rows.append({'label': label_name, **report[label_name]})
    return pd.DataFrame(rows)


def prediction_frame(indices, probabilities, candidate_name, stage):
    indices = np.asarray(indices, dtype=np.int64)
    labels = cached_labels[torch.from_numpy(indices)].numpy()
    probabilities = normalize_probability_matrix(probabilities)
    predictions = probabilities.argmax(axis=1)
    top_two = np.sort(probabilities, axis=1)[:, -2:]
    return pd.DataFrame({
        'row_index': indices,
        'image_id': fold_assignments.iloc[indices]['image_id'].astype(str).to_numpy(),
        'label': labels,
        'prediction': predictions,
        'prob_0': probabilities[:, 0],
        'prob_1': probabilities[:, 1],
        'prob_2': probabilities[:, 2],
        'confidence': top_two[:, 1],
        'top_two_margin': top_two[:, 1] - top_two[:, 0],
        'correct': labels == predictions,
        'fold': cached_folds[torch.from_numpy(indices)].numpy(),
        'candidate': candidate_name,
        'stage': stage,
    })


def specialist_prediction_frame(indices, probabilities, stage):
    indices = np.asarray(indices, dtype=np.int64)
    labels = cached_labels[torch.from_numpy(indices)].numpy()
    probabilities = normalize_probability_matrix(probabilities)
    binary_predictions = probabilities.argmax(axis=1)
    mapped_predictions = np.where(binary_predictions == 0, 0, 2)
    return pd.DataFrame({
        'row_index': indices,
        'image_id': fold_assignments.iloc[indices]['image_id'].astype(str).to_numpy(),
        'label': labels,
        'specialist_prediction': mapped_predictions,
        'prob_recyclable': probabilities[:, 0],
        'prob_organic': probabilities[:, 1],
        'in_specialist_label_scope': np.isin(labels, [0, 2]),
        'fold': cached_folds[torch.from_numpy(indices)].numpy(),
        'candidate': SPECIALIST_NAME,
        'stage': stage,
    })


def build_probe(kind):
    if kind == 'control':
        return V7RegularizedMLPFinal(), NUM_CLASSES, CONTROL_LABEL_SMOOTHING
    if kind == 'specialist':
        return ROSpecialistFinal(), 2, SPECIALIST_LABEL_SMOOTHING
    raise KeyError('Unknown probe kind: {}'.format(kind))


def predict_logits(model, indices, output_dim):
    index_list = [int(index) for index in indices]
    loader = DataLoader(CachedFinalRows(index_list), batch_size=PREDICT_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=PIN_MEMORY)
    logits_parts = []
    observed_indices = []
    model.eval()
    with torch.inference_mode():
        for batch_features, _batch_labels, batch_indices in loader:
            require(torch.isfinite(batch_features).all().item(), 'Prediction features contain NaN/Inf.')
            logits = model(batch_features.to(device=DEVICE, dtype=torch.float32, non_blocking=PIN_MEMORY))
            require(torch.isfinite(logits).all().item(), 'Probe logits contain NaN/Inf.')
            logits_parts.append(logits.cpu())
            observed_indices.extend(int(index) for index in batch_indices.tolist())
    require(observed_indices == index_list, 'Prediction order changed inside DataLoader.')
    logits = torch.cat(logits_parts).numpy().astype(np.float32)
    require(logits.shape == (len(index_list), output_dim), 'Probe logit shape is invalid.')
    return logits


def train_probe(kind, validation_fold, stage):
    require(kind in {'control', 'specialist'} and validation_fold in range(N_FOLDS), 'Unknown probe kind or validation fold.')
    fold_values = cached_folds.numpy()
    label_values = cached_labels.numpy()
    train_indices = np.flatnonzero(fold_values != validation_fold)
    valid_indices = np.flatnonzero(fold_values == validation_fold)
    if kind == 'specialist':
        train_indices = train_indices[np.isin(label_values[train_indices], [0, 2])]
        selection_valid_indices = valid_indices[np.isin(label_values[valid_indices], [0, 2])]
    else:
        selection_valid_indices = valid_indices
    require(len(train_indices) and len(selection_valid_indices), 'Probe partition is empty.')

    raw_train_labels = label_values[train_indices]
    train_labels = (raw_train_labels == 2).astype(np.int64) if kind == 'specialist' else raw_train_labels
    output_dim = 2 if kind == 'specialist' else NUM_CLASSES
    class_counts = np.bincount(train_labels, minlength=output_dim)
    require((class_counts > 0).all(), 'Probe training partition is missing a class.')
    class_weights = torch.tensor(class_counts.sum() / (output_dim * class_counts), dtype=torch.float32, device=DEVICE)

    seed_everything(SEED)
    generator = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(
        CachedFinalRows(train_indices),
        batch_size=HEAD_BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )
    model, output_dim, label_smoothing = build_probe(kind)
    model = model.to(DEVICE)
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    require(parameter_count == sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad), 'All probe parameters must be trainable.')
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smoothing)
    optimizer = torch.optim.AdamW(model.parameters(), lr=HEAD_LEARNING_RATE, weight_decay=HEAD_WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

    best_state = None
    best_epoch = None
    best_macro_f1 = -1.0
    best_log_loss = float('inf')
    stale_epochs = 0
    history = []
    training_started = time.perf_counter()
    selection_raw_labels = label_values[selection_valid_indices]
    selection_labels = (selection_raw_labels == 2).astype(np.int64) if kind == 'specialist' else selection_raw_labels
    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_started = time.perf_counter()
        model.train()
        loss_sum = 0.0
        seen = 0
        max_gradient_norm = 0.0
        learning_rate = optimizer.param_groups[0]['lr']
        for batch_features, batch_labels, _batch_indices in train_loader:
            require(torch.isfinite(batch_features).all().item(), 'Training features contain NaN/Inf.')
            batch_features = batch_features.to(device=DEVICE, dtype=torch.float32, non_blocking=PIN_MEMORY)
            batch_labels = batch_labels.to(device=DEVICE, non_blocking=PIN_MEMORY)
            if kind == 'specialist':
                batch_labels = (batch_labels == 2).long()
            optimizer.zero_grad(set_to_none=True)
            logits = model(batch_features)
            require(torch.isfinite(logits).all().item(), 'Training logits contain NaN/Inf.')
            loss = criterion(logits, batch_labels)
            require(torch.isfinite(loss).item(), 'Probe loss is non-finite.')
            loss.backward()
            gradient_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_MAX_NORM)
            require(torch.isfinite(gradient_norm).item(), 'Probe gradient norm is non-finite.')
            max_gradient_norm = max(max_gradient_norm, float(gradient_norm.item()))
            optimizer.step()
            require(all(torch.isfinite(parameter).all().item() for parameter in model.parameters()), 'Probe parameters became non-finite.')
            loss_sum += loss.item() * len(batch_labels)
            seen += len(batch_labels)
        scheduler.step()
        valid_logits = predict_logits(model, selection_valid_indices, output_dim)
        valid_probabilities = normalize_probabilities_from_logits(valid_logits)
        if kind == 'specialist':
            valid_metrics = binary_metric_bundle(selection_labels, valid_probabilities)
            valid_log_loss = valid_metrics['binary_log_loss']
        else:
            valid_metrics = metric_bundle(selection_labels, valid_probabilities)
            valid_log_loss = valid_metrics['multiclass_log_loss']
        valid_macro_f1 = valid_metrics['macro_f1']
        history.append({
            'stage': stage,
            'probe_kind': kind,
            'validation_fold': validation_fold,
            'seed': SEED,
            'epoch': epoch,
            'train_loss': loss_sum / seen,
            'valid_macro_f1': valid_macro_f1,
            'valid_log_loss': valid_log_loss,
            'learning_rate': learning_rate,
            'max_preclip_gradient_norm': max_gradient_norm,
            'epoch_seconds': time.perf_counter() - epoch_started,
        })
        print('{} {} fold={} epoch={:02d} loss={:.6f} f1={:.6f} logloss={:.6f}'.format(stage, kind, validation_fold, epoch, loss_sum / seen, valid_macro_f1, valid_log_loss))
        improved = valid_macro_f1 > best_macro_f1 + 1e-8 or (abs(valid_macro_f1 - best_macro_f1) <= 1e-8 and valid_log_loss < best_log_loss)
        if improved:
            best_macro_f1 = float(valid_macro_f1)
            best_log_loss = float(valid_log_loss)
            best_epoch = epoch
            best_state = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= EARLY_STOPPING_PATIENCE:
                break
    require(best_state is not None and best_epoch is not None, 'Probe training produced no valid checkpoint.')
    model.load_state_dict(best_state, strict=True)
    full_valid_logits = predict_logits(model, valid_indices, output_dim)
    full_valid_probabilities = normalize_probabilities_from_logits(full_valid_logits)
    if kind == 'specialist':
        in_scope = np.isin(label_values[valid_indices], [0, 2])
        binary_labels = (label_values[valid_indices][in_scope] == 2).astype(np.int64)
        metrics = binary_metric_bundle(binary_labels, full_valid_probabilities[in_scope])
        candidate_name = SPECIALIST_NAME
        prediction_table = specialist_prediction_frame(valid_indices, full_valid_probabilities, stage)
        report_labels = ('Recyclable', 'Organic')
    else:
        metrics = metric_bundle(label_values[valid_indices], full_valid_probabilities)
        candidate_name = CONTROL_NAME
        prediction_table = prediction_frame(valid_indices, full_valid_probabilities, candidate_name, stage)
        report_labels = CLASS_NAMES
    metrics.update({
        'schema_version': V8_SCHEMA_VERSION,
        'revision_marker': V8_REVISION_MARKER,
        'candidate': candidate_name,
        'probe_kind': kind,
        'stage': stage,
        'validation_fold': validation_fold,
        'training_folds': sorted(set(range(N_FOLDS)) - {validation_fold}),
        'seed': SEED,
        'best_epoch': best_epoch,
        'best_selection_macro_f1': best_macro_f1,
        'best_selection_log_loss': best_log_loss,
        'training_seconds': time.perf_counter() - training_started,
        'parameter_count': parameter_count,
        'train_class_counts': class_counts.tolist(),
        'class_weights': class_weights.detach().cpu().tolist(),
        'cache_signature': CACHE_SIGNATURE,
        'fold_assignment_sha256': FOLD_SIGNATURE,
        'locked_v8_contract_signature': LOCKED_V8_CONTRACT_SIGNATURE,
    })
    run_stem = '{}_{}_fold{}_seed{}'.format(stage, candidate_name, validation_fold, SEED)
    history_path = HEADS_DIR / (run_stem + '_history.csv')
    report_path = HEADS_DIR / (run_stem + '_classification_report.csv')
    prediction_path = HEADS_DIR / (run_stem + '_predictions.csv')
    metrics_path = HEADS_DIR / (run_stem + '_metrics.json')
    checkpoint_path = HEADS_DIR / (run_stem + '.pt')
    write_csv_atomic(pd.DataFrame(history), history_path)
    write_csv_atomic(report_frame(metrics['classification_report'], report_labels), report_path)
    write_csv_atomic(prediction_table, prediction_path)
    checkpoint_metadata = {
        'schema_version': V8_SCHEMA_VERSION,
        'revision_marker': V8_REVISION_MARKER,
        'candidate': candidate_name,
        'probe_kind': kind,
        'stage': stage,
        'validation_fold': validation_fold,
        'seed': SEED,
        'cache_signature': CACHE_SIGNATURE,
        'fold_assignment_sha256': FOLD_SIGNATURE,
        'model_id': MODEL_ID,
        'model_revision': MODEL_REVISION,
        'locked_v8_contract_signature': LOCKED_V8_CONTRACT_SIGNATURE,
        'max_epochs': MAX_EPOCHS,
        'patience': EARLY_STOPPING_PATIENCE,
        'learning_rate': HEAD_LEARNING_RATE,
        'weight_decay': HEAD_WEIGHT_DECAY,
        'gradient_clip_max_norm': GRAD_CLIP_MAX_NORM,
        'label_smoothing': label_smoothing,
        'dropout': CONTROL_DROPOUT if kind == 'control' else SPECIALIST_DROPOUT,
        'class_weights': class_weights.detach().cpu().tolist(),
        'checkpoint_selection': 'validation Macro F1, then lower log loss',
    }
    atomic_torch_save({'metadata': checkpoint_metadata, 'best_epoch': best_epoch, 'state_dict': best_state}, checkpoint_path)
    persisted = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    require(persisted.get('metadata') == checkpoint_metadata and persisted.get('best_epoch') == best_epoch, 'Saved probe checkpoint failed provenance validation.')
    metrics.update({
        'history_file': str(history_path),
        'classification_report_file': str(report_path),
        'prediction_file': str(prediction_path),
        'checkpoint_file': str(checkpoint_path),
    })
    write_json_atomic(metrics, metrics_path)
    result = {
        'metrics': metrics,
        'indices': valid_indices,
        'probabilities': full_valid_probabilities,
        'history': history,
        'prediction_table': prediction_table,
        'checkpoint_path': checkpoint_path,
    }
    del persisted, model, train_loader, criterion, optimizer, scheduler, best_state
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

## 8. V7 control reproduction

The exact V7 selected control is retrained from the cache on folds 1–4 and evaluated on fold 0. V8 stops before specialist training unless reproduced Macro F1 is within 0.0002 of the verified V7 artifact.

In [6]:
CONTROL_FOLD0 = train_probe('control', DEVELOPMENT_FOLD, 'fold0_control')
control_fold0_macro_f1 = CONTROL_FOLD0['metrics']['macro_f1']
control_reproduction_delta = abs(control_fold0_macro_f1 - V7_FOLD0_CONTROL_METRICS['macro_f1'])
CONTROL_REPRODUCTION = {
    'schema_version': V8_SCHEMA_VERSION,
    'revision_marker': V8_REVISION_MARKER,
    'pass': control_reproduction_delta <= CONTROL_REPRODUCTION_TOLERANCE,
    'candidate': CONTROL_NAME,
    'validation_fold': DEVELOPMENT_FOLD,
    'expected_v7_macro_f1': V7_FOLD0_CONTROL_METRICS['macro_f1'],
    'reproduced_macro_f1': control_fold0_macro_f1,
    'absolute_macro_f1_difference': control_reproduction_delta,
    'maximum_allowed_difference': CONTROL_REPRODUCTION_TOLERANCE,
    'resolved_v7_recipe': v7_control_checkpoint_metadata,
    'cache_signature': CACHE_SIGNATURE,
    'fold_assignment_sha256': FOLD_SIGNATURE,
    'checkpoint_file': CONTROL_FOLD0['metrics']['checkpoint_file'],
}
write_json_atomic(CONTROL_REPRODUCTION, ARTIFACT_DIR / 'control_reproduction.json')
if not CONTROL_REPRODUCTION['pass']:
    fail_closed('control_reproduction', 'Exact V7 selected control was not reproduced within the locked Macro-F1 tolerance.', CONTROL_REPRODUCTION)
print('Control reproduction passed; absolute Macro-F1 difference:', control_reproduction_delta)

fold0_control control fold=0 epoch=01 loss=0.285370 f1=0.985233 logloss=0.097650
fold0_control control fold=0 epoch=02 loss=0.234934 f1=0.985979 logloss=0.092178
fold0_control control fold=0 epoch=03 loss=0.226676 f1=0.987797 logloss=0.095111
fold0_control control fold=0 epoch=04 loss=0.222679 f1=0.988550 logloss=0.093349
fold0_control control fold=0 epoch=05 loss=0.219851 f1=0.989918 logloss=0.101128
fold0_control control fold=0 epoch=06 loss=0.219188 f1=0.988852 logloss=0.090078
fold0_control control fold=0 epoch=07 loss=0.218222 f1=0.989019 logloss=0.097577
fold0_control control fold=0 epoch=08 loss=0.217018 f1=0.989471 logloss=0.092065
fold0_control control fold=0 epoch=09 loss=0.216439 f1=0.989002 logloss=0.093820
fold0_control control fold=0 epoch=10 loss=0.215972 f1=0.989448 logloss=0.095453
Control reproduction passed; absolute Macro-F1 difference: 0.0


## 9. Fold-0 V8 screening

One specialist is trained on fold-0 development. Both predeclared fusion rules share that checkpoint. The fusion proposal is rejected per sample whenever it would change the main head's Electronic/non-Electronic decision, so Electronic predictions are invariant by construction.

In [7]:
def fuse_ro_probabilities(main_probabilities, specialist_probabilities, specialist_weight):
    main_probabilities = normalize_probability_matrix(main_probabilities)
    specialist_probabilities = normalize_probability_matrix(specialist_probabilities)
    require(main_probabilities.shape[0] == specialist_probabilities.shape[0], 'Main/specialist rows are misaligned.')
    require(main_probabilities.shape[1] == 3 and specialist_probabilities.shape[1] == 2, 'Main/specialist probability shapes are invalid.')
    ro_mass = main_probabilities[:, [0, 2]].sum(axis=1, keepdims=True)
    require((ro_mass > 0).all(), 'Main head has a zero R/O probability mass row.')
    main_ro_conditional = main_probabilities[:, [0, 2]] / ro_mass
    fused_ro_conditional = (1.0 - specialist_weight) * main_ro_conditional + specialist_weight * specialist_probabilities
    fused_ro_conditional = normalize_probability_matrix(fused_ro_conditional)
    proposal = np.empty_like(main_probabilities, dtype=np.float64)
    proposal[:, 1] = main_probabilities[:, 1]
    proposal[:, [0, 2]] = (1.0 - main_probabilities[:, 1:2]) * fused_ro_conditional
    proposal = normalize_probability_matrix(proposal)
    main_predictions = main_probabilities.argmax(axis=1)
    proposal_predictions = proposal.argmax(axis=1)
    preserve_main_row = (main_predictions == 1) | (proposal_predictions == 1)
    fused = proposal.copy()
    fused[preserve_main_row] = main_probabilities[preserve_main_row]
    fused = normalize_probability_matrix(fused)
    fused_predictions = fused.argmax(axis=1)
    require(np.array_equal(main_predictions == 1, fused_predictions == 1), 'Fusion changed the Electronic decision mask.')
    require(np.allclose(main_probabilities[:, 1], fused[:, 1], atol=1e-12, rtol=0.0), 'Fusion changed the main Electronic probability.')
    audit = {
        'specialist_weight': specialist_weight,
        'rows': len(fused),
        'rows_receiving_ro_redistribution': int((~preserve_main_row).sum()),
        'rows_preserved_to_protect_electronic_decision': int(preserve_main_row.sum()),
        'electronic_probability_max_abs_difference': float(np.abs(main_probabilities[:, 1] - fused[:, 1]).max()),
        'electronic_decision_changes': int(np.count_nonzero((main_predictions == 1) != (fused_predictions == 1))),
    }
    return fused, audit


def persist_fused_candidate(stage, validation_fold, candidate_name, control_result, specialist_result):
    spec = FUSION_SPECS[candidate_name]
    require(np.array_equal(control_result['indices'], specialist_result['indices']), 'Control/specialist validation indices differ.')
    probabilities, fusion_audit = fuse_ro_probabilities(
        control_result['probabilities'],
        specialist_result['probabilities'],
        spec['specialist_weight'],
    )
    indices = control_result['indices']
    labels = cached_labels[torch.from_numpy(indices)].numpy()
    metrics = metric_bundle(labels, probabilities)
    metrics.update({
        'schema_version': V8_SCHEMA_VERSION,
        'revision_marker': V8_REVISION_MARKER,
        'candidate': candidate_name,
        'candidate_spec': spec,
        'stage': stage,
        'validation_fold': validation_fold,
        'seed': SEED,
        'cache_signature': CACHE_SIGNATURE,
        'fold_assignment_sha256': FOLD_SIGNATURE,
        'locked_v8_contract_signature': LOCKED_V8_CONTRACT_SIGNATURE,
        'control_checkpoint': str(control_result['checkpoint_path']),
        'specialist_checkpoint': str(specialist_result['checkpoint_path']),
        'fusion_audit': fusion_audit,
    })
    run_stem = '{}_{}_fold{}_seed{}'.format(stage, candidate_name, validation_fold, SEED)
    prediction_path = HEADS_DIR / (run_stem + '_predictions.csv')
    report_path = HEADS_DIR / (run_stem + '_classification_report.csv')
    history_path = HEADS_DIR / (run_stem + '_history.csv')
    metrics_path = HEADS_DIR / (run_stem + '_metrics.json')
    checkpoint_path = HEADS_DIR / (run_stem + '.pt')
    predictions = prediction_frame(indices, probabilities, candidate_name, stage)
    fusion_history = pd.DataFrame(specialist_result['history']).assign(fusion_candidate=candidate_name, specialist_weight=spec['specialist_weight'])
    write_csv_atomic(predictions, prediction_path)
    write_csv_atomic(report_frame(metrics['classification_report'], CLASS_NAMES), report_path)
    write_csv_atomic(fusion_history, history_path)
    descriptor_metadata = {
        'schema_version': V8_SCHEMA_VERSION,
        'revision_marker': V8_REVISION_MARKER,
        'candidate': candidate_name,
        'candidate_spec': spec,
        'stage': stage,
        'validation_fold': validation_fold,
        'seed': SEED,
        'cache_signature': CACHE_SIGNATURE,
        'fold_assignment_sha256': FOLD_SIGNATURE,
        'model_id': MODEL_ID,
        'model_revision': MODEL_REVISION,
        'locked_v8_contract_signature': LOCKED_V8_CONTRACT_SIGNATURE,
        'control_checkpoint': str(control_result['checkpoint_path']),
        'specialist_checkpoint': str(specialist_result['checkpoint_path']),
        'fusion_invariants': LOCKED_V8_CONTRACT['fusion_invariants'],
    }
    atomic_torch_save({'metadata': descriptor_metadata}, checkpoint_path)
    persisted = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    require(persisted.get('metadata') == descriptor_metadata, 'Fusion checkpoint descriptor failed provenance validation.')
    metrics.update({
        'prediction_file': str(prediction_path),
        'classification_report_file': str(report_path),
        'history_file': str(history_path),
        'checkpoint_file': str(checkpoint_path),
    })
    write_json_atomic(metrics, metrics_path)
    return {
        'metrics': metrics,
        'indices': indices,
        'probabilities': probabilities,
        'prediction_table': predictions,
        'fusion_audit': fusion_audit,
        'checkpoint_path': checkpoint_path,
    }


def paired_counts(labels, control_probabilities, candidate_probabilities):
    labels = np.asarray(labels, dtype=np.int64)
    control_predictions = normalize_probability_matrix(control_probabilities).argmax(axis=1)
    candidate_predictions = normalize_probability_matrix(candidate_probabilities).argmax(axis=1)
    control_correct = control_predictions == labels
    candidate_correct = candidate_predictions == labels
    return {
        'corrected_samples': int((~control_correct & candidate_correct).sum()),
        'regressed_samples': int((control_correct & ~candidate_correct).sum()),
        'net_corrections': int((~control_correct & candidate_correct).sum() - (control_correct & ~candidate_correct).sum()),
        'prediction_disagreements': int((control_predictions != candidate_predictions).sum()),
    }


def evaluate_fold0_gate(candidate_result, control_result):
    metrics = candidate_result['metrics']
    control_metrics = control_result['metrics']
    labels = cached_labels[torch.from_numpy(candidate_result['indices'])].numpy()
    paired = paired_counts(labels, control_result['probabilities'], candidate_result['probabilities'])
    ro_errors = metrics['recyclable_to_organic'] + metrics['organic_to_recyclable']
    control_ro_errors = control_metrics['recyclable_to_organic'] + control_metrics['organic_to_recyclable']
    electronic_f1 = metrics['classification_report']['Electronic']['f1-score']
    control_electronic_f1 = control_metrics['classification_report']['Electronic']['f1-score']
    conditions = {
        'macro_f1': metrics['macro_f1'] >= FOLD0_GATES['minimum_macro_f1'],
        'gain_over_control': metrics['macro_f1'] - control_metrics['macro_f1'] >= FOLD0_GATES['minimum_gain_over_reproduced_control'],
        'ro_error_reduction': control_ro_errors - ro_errors >= FOLD0_GATES['minimum_ro_error_reduction'],
        'total_errors': metrics['total_errors'] - control_metrics['total_errors'] <= FOLD0_GATES['maximum_total_error_increase'],
        'electronic_f1': electronic_f1 >= control_electronic_f1 - FOLD0_GATES['maximum_electronic_f1_degradation'],
        'corrected_samples': paired['corrected_samples'] >= FOLD0_GATES['minimum_corrected_samples'],
        'not_single_sample_dependent': paired['net_corrections'] >= FOLD0_GATES['minimum_net_corrections'],
        'locked_contract_unchanged': sha256_json(LOCKED_V8_CONTRACT) == LOCKED_V8_CONTRACT_SIGNATURE,
        'electronic_decision_invariant': candidate_result['fusion_audit']['electronic_decision_changes'] == 0,
    }
    return {
        'pass': all(conditions.values()),
        'conditions': conditions,
        'observed': {
            'macro_f1': metrics['macro_f1'],
            'gain_over_control': metrics['macro_f1'] - control_metrics['macro_f1'],
            'ro_errors': ro_errors,
            'ro_error_reduction': control_ro_errors - ro_errors,
            'total_errors': metrics['total_errors'],
            'total_error_delta': metrics['total_errors'] - control_metrics['total_errors'],
            'electronic_f1': electronic_f1,
            'electronic_f1_delta': electronic_f1 - control_electronic_f1,
            **paired,
        },
        'gates': FOLD0_GATES,
    }


SPECIALIST_FOLD0 = train_probe('specialist', DEVELOPMENT_FOLD, 'fold0_specialist')
FOLD0_CANDIDATE_RESULTS = {
    candidate_name: persist_fused_candidate('fold0', DEVELOPMENT_FOLD, candidate_name, CONTROL_FOLD0, SPECIALIST_FOLD0)
    for candidate_name in FUSION_SPECS
}
FOLD0_GATE_RESULTS = {
    candidate_name: evaluate_fold0_gate(result, CONTROL_FOLD0)
    for candidate_name, result in FOLD0_CANDIDATE_RESULTS.items()
}
candidate_order = {name: position for position, name in enumerate(FUSION_SPECS)}
ranked_candidates = sorted(
    FOLD0_CANDIDATE_RESULTS,
    key=lambda name: (
        -FOLD0_CANDIDATE_RESULTS[name]['metrics']['macro_f1'],
        FOLD0_CANDIDATE_RESULTS[name]['metrics']['multiclass_log_loss'],
        candidate_order[name],
    ),
)
passing_candidates = [name for name in ranked_candidates if FOLD0_GATE_RESULTS[name]['pass']]
SELECTED_V8_CANDIDATE = passing_candidates[0] if passing_candidates else None
DIAGNOSTIC_CANDIDATE = SELECTED_V8_CANDIDATE or ranked_candidates[0]
RUN_CONFIRMATION = SELECTED_V8_CANDIDATE is not None

comparison_rows = [{
    'candidate': CONTROL_NAME,
    'role': 'reproduced_control',
    'macro_f1': CONTROL_FOLD0['metrics']['macro_f1'],
    'macro_f1_delta': 0.0,
    'accuracy': CONTROL_FOLD0['metrics']['accuracy'],
    'multiclass_log_loss': CONTROL_FOLD0['metrics']['multiclass_log_loss'],
    'recyclable_f1': CONTROL_FOLD0['metrics']['classification_report']['Recyclable']['f1-score'],
    'electronic_f1': CONTROL_FOLD0['metrics']['classification_report']['Electronic']['f1-score'],
    'organic_f1': CONTROL_FOLD0['metrics']['classification_report']['Organic']['f1-score'],
    'recyclable_to_organic': CONTROL_FOLD0['metrics']['recyclable_to_organic'],
    'organic_to_recyclable': CONTROL_FOLD0['metrics']['organic_to_recyclable'],
    'total_errors': CONTROL_FOLD0['metrics']['total_errors'],
    'gate_pass': True,
}]
for candidate_name in FUSION_SPECS:
    metrics = FOLD0_CANDIDATE_RESULTS[candidate_name]['metrics']
    comparison_rows.append({
        'candidate': candidate_name,
        'role': FUSION_SPECS[candidate_name]['role'],
        'macro_f1': metrics['macro_f1'],
        'macro_f1_delta': metrics['macro_f1'] - CONTROL_FOLD0['metrics']['macro_f1'],
        'accuracy': metrics['accuracy'],
        'multiclass_log_loss': metrics['multiclass_log_loss'],
        'recyclable_f1': metrics['classification_report']['Recyclable']['f1-score'],
        'electronic_f1': metrics['classification_report']['Electronic']['f1-score'],
        'organic_f1': metrics['classification_report']['Organic']['f1-score'],
        'recyclable_to_organic': metrics['recyclable_to_organic'],
        'organic_to_recyclable': metrics['organic_to_recyclable'],
        'total_errors': metrics['total_errors'],
        'gate_pass': FOLD0_GATE_RESULTS[candidate_name]['pass'],
    })
FOLD0_COMPARISON = pd.DataFrame(comparison_rows)
write_csv_atomic(FOLD0_COMPARISON, ARTIFACT_DIR / 'fold0_candidate_comparison.csv')
write_json_atomic({
    'control': CONTROL_FOLD0['metrics'],
    'specialist': SPECIALIST_FOLD0['metrics'],
    'candidates': {name: result['metrics'] for name, result in FOLD0_CANDIDATE_RESULTS.items()},
}, ARTIFACT_DIR / 'fold0_candidate_metrics.json')
FOLD0_DECISION = {
    'schema_version': V8_SCHEMA_VERSION,
    'revision_marker': V8_REVISION_MARKER,
    'pass': RUN_CONFIRMATION,
    'selected_candidate': SELECTED_V8_CANDIDATE,
    'diagnostic_candidate': DIAGNOSTIC_CANDIDATE,
    'candidate_ranking': ranked_candidates,
    'gate_results': FOLD0_GATE_RESULTS,
    'locked_contract_signature': LOCKED_V8_CONTRACT_SIGNATURE,
    'cache_signature': CACHE_SIGNATURE,
    'fold_assignment_sha256': FOLD_SIGNATURE,
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'selected_approach': APPROACH_DECISION['selected_family'],
    'fixed_hyperparameters': LOCKED_V8_CONTRACT['training'],
    'gates': FOLD0_GATES,
    'reason': 'At least one locked specialist fusion passed every fold-0 gate; run the selected design on confirmation folds.' if RUN_CONFIRMATION else 'No locked specialist fusion passed every fold-0 gate; stop cleanly and retain V7.',
    'historical_folds1_4_accessed_for_v8_direction': True,
    'official_test_accessed': False,
}
write_json_atomic(FOLD0_DECISION, ARTIFACT_DIR / 'fold0_decision.json')
if not RUN_CONFIRMATION:
    FINAL_ADOPTION_DECISION = {
        'schema_version': V8_SCHEMA_VERSION,
        'revision_marker': V8_REVISION_MARKER,
        'adopt_v8': False,
        'final_pipeline': 'V7 regularized_mlp_final',
        'reason': FOLD0_DECISION['reason'],
        'fold0_decision': FOLD0_DECISION,
        'confirmation_ran': False,
        'cache_signature': CACHE_SIGNATURE,
        'fold_assignment_sha256': FOLD_SIGNATURE,
        'model_id': MODEL_ID,
        'model_revision': MODEL_REVISION,
        'selected_approach': APPROACH_DECISION['selected_family'],
        'fixed_hyperparameters': LOCKED_V8_CONTRACT['training'],
        'gates': {'fold0': FOLD0_GATES, 'confirmation': CONFIRMATION_GATES},
        'official_test_accessed': False,
        'submission_created': False,
    }
    write_json_atomic(FINAL_ADOPTION_DECISION, ARTIFACT_DIR / 'final_adoption_decision.json')
print(FOLD0_COMPARISON.to_string(index=False))
print('Fold-0 gate:', FOLD0_DECISION['reason'])

fold0_specialist specialist fold=0 epoch=01 loss=0.179061 f1=0.982746 logloss=0.067824
fold0_specialist specialist fold=0 epoch=02 loss=0.141653 f1=0.984305 logloss=0.060784
fold0_specialist specialist fold=0 epoch=03 loss=0.131749 f1=0.984972 logloss=0.060169
fold0_specialist specialist fold=0 epoch=04 loss=0.128440 f1=0.985193 logloss=0.059494
fold0_specialist specialist fold=0 epoch=05 loss=0.126412 f1=0.986544 logloss=0.059331
fold0_specialist specialist fold=0 epoch=06 loss=0.124874 f1=0.986094 logloss=0.056823
fold0_specialist specialist fold=0 epoch=07 loss=0.123946 f1=0.985869 logloss=0.059939
fold0_specialist specialist fold=0 epoch=08 loss=0.123465 f1=0.986314 logloss=0.059581
fold0_specialist specialist fold=0 epoch=09 loss=0.123183 f1=0.986321 logloss=0.058569
fold0_specialist specialist fold=0 epoch=10 loss=0.122589 f1=0.986543 logloss=0.060589
            candidate                                role  macro_f1  macro_f1_delta  accuracy  multiclass_log_loss  recyclable_f1 

## 10. Paired diagnostics and gate decision

Diagnostics compare the best passing candidate—or the best-ranked rejected candidate when none passes—with the reproduced control. Image IDs are emitted only for traceable manual review; no sample is relabeled or removed.

In [8]:
def paired_diagnostics(prefix, indices, control_probabilities, candidate_probabilities, candidate_name, specialist_probabilities=None):
    indices = np.asarray(indices, dtype=np.int64)
    labels = cached_labels[torch.from_numpy(indices)].numpy()
    control_probabilities = normalize_probability_matrix(control_probabilities)
    candidate_probabilities = normalize_probability_matrix(candidate_probabilities)
    if specialist_probabilities is not None:
        specialist_probabilities = normalize_probability_matrix(specialist_probabilities)
        require(specialist_probabilities.shape == (len(indices), 2), 'Specialist diagnostic probability shape is invalid.')
    control_predictions = control_probabilities.argmax(axis=1)
    candidate_predictions = candidate_probabilities.argmax(axis=1)
    control_correct = control_predictions == labels
    candidate_correct = candidate_predictions == labels
    control_top_two = np.sort(control_probabilities, axis=1)[:, -2:]
    candidate_top_two = np.sort(candidate_probabilities, axis=1)[:, -2:]
    status = np.select(
        [~control_correct & candidate_correct, control_correct & ~candidate_correct, control_correct & candidate_correct],
        ['corrected', 'regressed', 'unchanged_correct'],
        default='unchanged_error',
    )
    paired = pd.DataFrame({
        'row_index': indices,
        'image_id': fold_assignments.iloc[indices]['image_id'].astype(str).to_numpy(),
        'label': labels,
        'fold': cached_folds[torch.from_numpy(indices)].numpy(),
        'control_prediction': control_predictions,
        'candidate_prediction': candidate_predictions,
        'control_confidence': control_top_two[:, 1],
        'candidate_confidence': candidate_top_two[:, 1],
        'control_margin': control_top_two[:, 1] - control_top_two[:, 0],
        'candidate_margin': candidate_top_two[:, 1] - candidate_top_two[:, 0],
        'status': status,
        'manual_review_candidate': (~candidate_correct) & (candidate_top_two[:, 1] >= HIGH_CONFIDENCE_ERROR_THRESHOLD),
        'candidate': candidate_name,
    })
    write_csv_atomic(paired, ARTIFACT_DIR / (prefix + '_paired_error_delta.csv'))
    write_csv_atomic(paired.loc[paired['status'] == 'corrected'], ARTIFACT_DIR / (prefix + '_corrected_samples.csv'))
    write_csv_atomic(paired.loc[paired['status'] == 'regressed'], ARTIFACT_DIR / (prefix + '_regressed_samples.csv'))
    write_csv_atomic(
        paired.loc[~candidate_correct & (candidate_top_two[:, 1] >= HIGH_CONFIDENCE_ERROR_THRESHOLD)].sort_values('candidate_confidence', ascending=False),
        ARTIFACT_DIR / (prefix + '_high_confidence_errors.csv'),
    )

    disagreement = pd.crosstab(
        pd.Series(control_predictions, name='control_prediction'),
        pd.Series(candidate_predictions, name='candidate_prediction'),
    ).reindex(index=range(NUM_CLASSES), columns=range(NUM_CLASSES), fill_value=0)
    write_csv_atomic(disagreement.reset_index(), ARTIFACT_DIR / (prefix + '_prediction_disagreement_matrix.csv'))

    confidence_bin = pd.cut(control_top_two[:, 1], bins=CONFIDENCE_BINS, right=False, include_lowest=True)
    confidence_rows = []
    for interval in confidence_bin.categories:
        mask = confidence_bin == interval
        confidence_rows.append({
            'control_confidence_bin': str(interval),
            'rows': int(mask.sum()),
            'control_errors': int((~control_correct & mask).sum()),
            'candidate_errors': int((~candidate_correct & mask).sum()),
            'corrected': int((~control_correct & candidate_correct & mask).sum()),
            'regressed': int((control_correct & ~candidate_correct & mask).sum()),
        })
    write_csv_atomic(pd.DataFrame(confidence_rows), ARTIFACT_DIR / (prefix + '_control_confidence_bins.csv'))

    control_matrix = confusion_matrix(labels, control_predictions, labels=list(range(NUM_CLASSES)))
    candidate_matrix = confusion_matrix(labels, candidate_predictions, labels=list(range(NUM_CLASSES)))
    figure, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for axis, matrix, title in zip(axes, (control_matrix, candidate_matrix), ('reproduced control', candidate_name)):
        image = axis.imshow(matrix, cmap='Blues')
        axis.set_title(title)
        axis.set_xlabel('prediction')
        axis.set_ylabel('label')
        axis.set_xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=25)
        axis.set_yticks(range(NUM_CLASSES), CLASS_NAMES)
        for row in range(NUM_CLASSES):
            for column in range(NUM_CLASSES):
                axis.text(column, row, str(matrix[row, column]), ha='center', va='center')
        figure.colorbar(image, ax=axis, fraction=0.046)
    figure.tight_layout()
    figure.savefig(ARTIFACT_DIR / (prefix + '_confusion_comparison.png'), dpi=160, bbox_inches='tight')
    plt.close(figure)

    figure, axes = plt.subplots(2, 2, figsize=(11, 8))
    axes[0, 0].hist(control_top_two[:, 1], bins=40, alpha=0.8)
    axes[0, 0].set_title('control confidence')
    axes[0, 1].hist(candidate_top_two[:, 1], bins=40, alpha=0.8)
    axes[0, 1].set_title('{} confidence'.format(candidate_name))
    axes[1, 0].hist(control_top_two[:, 1] - control_top_two[:, 0], bins=40, alpha=0.8)
    axes[1, 0].set_title('control top-two margin')
    axes[1, 1].hist(candidate_top_two[:, 1] - candidate_top_two[:, 0], bins=40, alpha=0.8)
    axes[1, 1].set_title('{} top-two margin'.format(candidate_name))
    for axis in axes.flat:
        axis.set_ylabel('rows')
    figure.tight_layout()
    figure.savefig(ARTIFACT_DIR / (prefix + '_confidence_margin_comparison.png'), dpi=160, bbox_inches='tight')
    plt.close(figure)

    electronic_mask = labels == 1
    diagnostic_summary = {
        'schema_version': V8_SCHEMA_VERSION,
        'revision_marker': V8_REVISION_MARKER,
        'candidate': candidate_name,
        'rows': len(indices),
        'corrected_samples': int((status == 'corrected').sum()),
        'regressed_samples': int((status == 'regressed').sum()),
        'unchanged_errors': int((status == 'unchanged_error').sum()),
        'prediction_disagreement_matrix': disagreement.to_dict(),
        'control_confidence': confidence_summary(control_top_two[:, 1]),
        'candidate_confidence': confidence_summary(candidate_top_two[:, 1]),
        'control_margin': confidence_summary(control_top_two[:, 1] - control_top_two[:, 0]),
        'candidate_margin': confidence_summary(candidate_top_two[:, 1] - candidate_top_two[:, 0]),
        'electronic_behavior': {
            'true_electronic_rows': int(electronic_mask.sum()),
            'electronic_prediction_changes': int(np.count_nonzero((control_predictions == 1) != (candidate_predictions == 1))),
            'true_electronic_any_class_prediction_changes': int(np.count_nonzero(control_predictions[electronic_mask] != candidate_predictions[electronic_mask])),
            'control_prediction_counts_on_true_electronic': np.bincount(control_predictions[electronic_mask], minlength=NUM_CLASSES).tolist(),
            'candidate_prediction_counts_on_true_electronic': np.bincount(candidate_predictions[electronic_mask], minlength=NUM_CLASSES).tolist(),
            'electronic_probability_max_abs_difference': float(np.abs(control_probabilities[:, 1] - candidate_probabilities[:, 1]).max()),
            'specialist_recyclable_probability_on_true_electronic': confidence_summary(specialist_probabilities[electronic_mask, 0]) if specialist_probabilities is not None else None,
            'specialist_organic_probability_on_true_electronic': confidence_summary(specialist_probabilities[electronic_mask, 1]) if specialist_probabilities is not None else None,
            'specialist_binary_predictions_on_true_electronic': {
                'Recyclable': int((specialist_probabilities[electronic_mask].argmax(axis=1) == 0).sum()),
                'Organic': int((specialist_probabilities[electronic_mask].argmax(axis=1) == 1).sum()),
            } if specialist_probabilities is not None else None,
        },
        'manual_review_policy': 'High-confidence residual errors are candidates for human inspection only; official labels remain unchanged.',
    }
    write_json_atomic(diagnostic_summary, ARTIFACT_DIR / (prefix + '_diagnostic_summary.json'))
    return paired, diagnostic_summary


fold0_diagnostic_result = FOLD0_CANDIDATE_RESULTS[DIAGNOSTIC_CANDIDATE]
FOLD0_PAIRED_DELTA, FOLD0_DIAGNOSTIC_SUMMARY = paired_diagnostics(
    'fold0',
    fold0_diagnostic_result['indices'],
    CONTROL_FOLD0['probabilities'],
    fold0_diagnostic_result['probabilities'],
    DIAGNOSTIC_CANDIDATE,
    SPECIALIST_FOLD0['probabilities'],
)
print(json.dumps(FOLD0_DECISION, indent=2))

{
  "schema_version": 1,
  "revision_marker": "v8_ro_hierarchical_specialist_final_cache_v1",
  "pass": false,
  "selected_candidate": null,
  "diagnostic_candidate": "ro_specialist_full",
  "candidate_ranking": [
    "ro_specialist_full",
    "ro_specialist_blend75"
  ],
  "gate_results": {
    "ro_specialist_blend75": {
      "pass": false,
      "conditions": {
        "macro_f1": false,
        "gain_over_control": false,
        "ro_error_reduction": false,
        "total_errors": false,
        "electronic_f1": true,
        "corrected_samples": true,
        "not_single_sample_dependent": false,
        "locked_contract_unchanged": true,
        "electronic_decision_invariant": true
      },
      "observed": {
        "macro_f1": 0.9893223868666294,
        "gain_over_control": -0.0005956249239723599,
        "ro_errors": 58,
        "ro_error_reduction": -4,
        "total_errors": 65,
        "total_error_delta": 4,
        "electronic_f1": 0.9955891619407687,
        "electr

## 11. Conditional folds 1–4 confirmation

This section runs only when a locked fusion passes every fold-0 gate. The same fusion weight, architectures, recipe, seed, and thresholds are reused without retuning. These are historical assessment folds reused for cross-fold confirmation; the official test remains untouched.

In [9]:
CONFIRMATION_RAN = False
CONFIRMATION_STATUS = {
    'ran': False,
    'reason': 'Fold-0 gate did not pass.',
    'cache_signature': CACHE_SIGNATURE,
    'fold_assignment_sha256': FOLD_SIGNATURE,
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'selected_approach': APPROACH_DECISION['selected_family'],
    'fixed_hyperparameters': LOCKED_V8_CONTRACT['training'],
    'gates': CONFIRMATION_GATES,
}
if RUN_CONFIRMATION:
    CONFIRMATION_RAN = True
    confirmation_control_parts = []
    confirmation_candidate_parts = []
    confirmation_specialist_parts = []
    confirmation_indices_parts = []
    confirmation_per_fold_rows = []
    selected_spec = FUSION_SPECS[SELECTED_V8_CANDIDATE]
    for validation_fold in CONFIRMATION_FOLDS:
        control_result = train_probe('control', validation_fold, 'confirmation_control')
        specialist_result = train_probe('specialist', validation_fold, 'confirmation_specialist')
        candidate_result = persist_fused_candidate('confirmation', validation_fold, SELECTED_V8_CANDIDATE, control_result, specialist_result)
        control_metrics = control_result['metrics']
        candidate_metrics = candidate_result['metrics']
        fold_labels = cached_labels[torch.from_numpy(candidate_result['indices'])].numpy()
        fold_paired = paired_counts(fold_labels, control_result['probabilities'], candidate_result['probabilities'])
        paired_diagnostics(
            'confirmation_fold{}'.format(validation_fold),
            candidate_result['indices'],
            control_result['probabilities'],
            candidate_result['probabilities'],
            SELECTED_V8_CANDIDATE,
            specialist_result['probabilities'],
        )
        confirmation_per_fold_rows.append({
            'validation_fold': validation_fold,
            'candidate': SELECTED_V8_CANDIDATE,
            'seed': SEED,
            'cache_signature': CACHE_SIGNATURE,
            'fold_assignment_sha256': FOLD_SIGNATURE,
            'model_revision': MODEL_REVISION,
            'control_macro_f1': control_metrics['macro_f1'],
            'candidate_macro_f1': candidate_metrics['macro_f1'],
            'macro_f1_delta': candidate_metrics['macro_f1'] - control_metrics['macro_f1'],
            'control_accuracy': control_metrics['accuracy'],
            'candidate_accuracy': candidate_metrics['accuracy'],
            'control_log_loss': control_metrics['multiclass_log_loss'],
            'candidate_log_loss': candidate_metrics['multiclass_log_loss'],
            'control_total_errors': control_metrics['total_errors'],
            'candidate_total_errors': candidate_metrics['total_errors'],
            'total_error_delta': candidate_metrics['total_errors'] - control_metrics['total_errors'],
            'control_recyclable_to_organic': control_metrics['recyclable_to_organic'],
            'candidate_recyclable_to_organic': candidate_metrics['recyclable_to_organic'],
            'control_organic_to_recyclable': control_metrics['organic_to_recyclable'],
            'candidate_organic_to_recyclable': candidate_metrics['organic_to_recyclable'],
            'control_ro_errors': control_metrics['recyclable_to_organic'] + control_metrics['organic_to_recyclable'],
            'candidate_ro_errors': candidate_metrics['recyclable_to_organic'] + candidate_metrics['organic_to_recyclable'],
            'control_recyclable_f1': control_metrics['classification_report']['Recyclable']['f1-score'],
            'candidate_recyclable_f1': candidate_metrics['classification_report']['Recyclable']['f1-score'],
            'control_electronic_f1': control_metrics['classification_report']['Electronic']['f1-score'],
            'candidate_electronic_f1': candidate_metrics['classification_report']['Electronic']['f1-score'],
            'control_organic_f1': control_metrics['classification_report']['Organic']['f1-score'],
            'candidate_organic_f1': candidate_metrics['classification_report']['Organic']['f1-score'],
            **fold_paired,
            'candidate_beats_control': candidate_metrics['macro_f1'] > control_metrics['macro_f1'],
        })
        confirmation_indices_parts.append(candidate_result['indices'])
        confirmation_control_parts.append(control_result['probabilities'])
        confirmation_candidate_parts.append(candidate_result['probabilities'])
        confirmation_specialist_parts.append(specialist_result['probabilities'])

    confirmation_indices = np.concatenate(confirmation_indices_parts)
    confirmation_control_probabilities = np.concatenate(confirmation_control_parts)
    confirmation_candidate_probabilities = np.concatenate(confirmation_candidate_parts)
    confirmation_specialist_probabilities = np.concatenate(confirmation_specialist_parts)
    order = np.argsort(confirmation_indices)
    confirmation_indices = confirmation_indices[order]
    confirmation_control_probabilities = confirmation_control_probabilities[order]
    confirmation_candidate_probabilities = confirmation_candidate_probabilities[order]
    confirmation_specialist_probabilities = confirmation_specialist_probabilities[order]
    require(len(confirmation_indices) == 21221 and len(np.unique(confirmation_indices)) == 21221, 'Confirmation rows are incomplete or duplicated.')
    require(set(cached_folds[torch.from_numpy(confirmation_indices)].numpy()) == set(CONFIRMATION_FOLDS), 'Confirmation fold coverage is invalid.')
    confirmation_labels = cached_labels[torch.from_numpy(confirmation_indices)].numpy()
    confirmation_control_metrics = metric_bundle(confirmation_labels, confirmation_control_probabilities)
    confirmation_candidate_metrics = metric_bundle(confirmation_labels, confirmation_candidate_probabilities)
    confirmation_selected_predictions = prediction_frame(
        confirmation_indices,
        confirmation_candidate_probabilities,
        SELECTED_V8_CANDIDATE,
        'confirmation_aggregate',
    )
    write_csv_atomic(pd.DataFrame(confirmation_per_fold_rows), ARTIFACT_DIR / 'confirmation_per_fold_comparison.csv')
    aggregate_control_row = {
        'role': 'reproduced_control',
        'candidate': CONTROL_NAME,
        'macro_f1': confirmation_control_metrics['macro_f1'],
        'macro_f1_delta': 0.0,
        'accuracy': confirmation_control_metrics['accuracy'],
        'multiclass_log_loss': confirmation_control_metrics['multiclass_log_loss'],
        'log_loss_delta': 0.0,
        'recyclable_f1': confirmation_control_metrics['classification_report']['Recyclable']['f1-score'],
        'electronic_f1': confirmation_control_metrics['classification_report']['Electronic']['f1-score'],
        'organic_f1': confirmation_control_metrics['classification_report']['Organic']['f1-score'],
        'recyclable_to_organic': confirmation_control_metrics['recyclable_to_organic'],
        'organic_to_recyclable': confirmation_control_metrics['organic_to_recyclable'],
        'total_errors': confirmation_control_metrics['total_errors'],
    }
    aggregate_candidate_row = {
        'role': 'v8_candidate',
        'candidate': SELECTED_V8_CANDIDATE,
        'macro_f1': confirmation_candidate_metrics['macro_f1'],
        'macro_f1_delta': confirmation_candidate_metrics['macro_f1'] - confirmation_control_metrics['macro_f1'],
        'accuracy': confirmation_candidate_metrics['accuracy'],
        'multiclass_log_loss': confirmation_candidate_metrics['multiclass_log_loss'],
        'log_loss_delta': confirmation_candidate_metrics['multiclass_log_loss'] - confirmation_control_metrics['multiclass_log_loss'],
        'recyclable_f1': confirmation_candidate_metrics['classification_report']['Recyclable']['f1-score'],
        'electronic_f1': confirmation_candidate_metrics['classification_report']['Electronic']['f1-score'],
        'organic_f1': confirmation_candidate_metrics['classification_report']['Organic']['f1-score'],
        'recyclable_to_organic': confirmation_candidate_metrics['recyclable_to_organic'],
        'organic_to_recyclable': confirmation_candidate_metrics['organic_to_recyclable'],
        'total_errors': confirmation_candidate_metrics['total_errors'],
    }
    for row in (aggregate_control_row, aggregate_candidate_row):
        row.update({'seed': SEED, 'cache_signature': CACHE_SIGNATURE, 'fold_assignment_sha256': FOLD_SIGNATURE, 'model_revision': MODEL_REVISION})
    write_csv_atomic(pd.DataFrame([aggregate_control_row, aggregate_candidate_row]), ARTIFACT_DIR / 'confirmation_aggregate_comparison.csv')
    write_csv_atomic(confirmation_selected_predictions, ARTIFACT_DIR / 'confirmation_selected_predictions.csv')
    write_json_atomic({
        'control': confirmation_control_metrics,
        'candidate': confirmation_candidate_metrics,
        'selected_candidate': SELECTED_V8_CANDIDATE,
        'candidate_spec': selected_spec,
        'seed': SEED,
        'cache_signature': CACHE_SIGNATURE,
        'fold_assignment_sha256': FOLD_SIGNATURE,
        'model_id': MODEL_ID,
        'model_revision': MODEL_REVISION,
        'fixed_hyperparameters': LOCKED_V8_CONTRACT['training'],
        'gates': CONFIRMATION_GATES,
    }, ARTIFACT_DIR / 'confirmation_aggregate_metrics.json')
    CONFIRMATION_PAIRED_DELTA, CONFIRMATION_DIAGNOSTIC_SUMMARY = paired_diagnostics(
        'confirmation',
        confirmation_indices,
        confirmation_control_probabilities,
        confirmation_candidate_probabilities,
        SELECTED_V8_CANDIDATE,
        confirmation_specialist_probabilities,
    )
    folds_beating_control = int(sum(row['candidate_beats_control'] for row in confirmation_per_fold_rows))
    candidate_ro_errors = confirmation_candidate_metrics['recyclable_to_organic'] + confirmation_candidate_metrics['organic_to_recyclable']
    candidate_electronic_f1 = confirmation_candidate_metrics['classification_report']['Electronic']['f1-score']
    control_electronic_f1 = confirmation_control_metrics['classification_report']['Electronic']['f1-score']
    confirmation_conditions = {
        'aggregate_macro_f1': confirmation_candidate_metrics['macro_f1'] >= CONFIRMATION_GATES['minimum_aggregate_macro_f1'],
        'folds_beating_control': folds_beating_control >= CONFIRMATION_GATES['minimum_folds_beating_reproduced_control'],
        'ro_errors_below_verified_v7': candidate_ro_errors < CONFIRMATION_GATES['maximum_ro_errors_exclusive'],
        'electronic_f1': candidate_electronic_f1 >= CONFIRMATION_GATES['minimum_electronic_f1'] or candidate_electronic_f1 >= control_electronic_f1 - CONFIRMATION_GATES['maximum_electronic_f1_degradation'],
        'total_errors_decline': confirmation_candidate_metrics['total_errors'] < confirmation_control_metrics['total_errors'],
        'locked_design_unchanged': sha256_json(LOCKED_V8_CONTRACT) == LOCKED_V8_CONTRACT_SIGNATURE,
        'electronic_decision_invariant': CONFIRMATION_DIAGNOSTIC_SUMMARY['electronic_behavior']['electronic_prediction_changes'] == 0,
    }
    confirmation_pass = all(confirmation_conditions.values())
    CONFIRMATION_STATUS = {
        'ran': True,
        'pass': confirmation_pass,
        'reason': 'Every locked confirmation gate passed.' if confirmation_pass else 'At least one locked confirmation gate failed.',
        'selected_candidate': SELECTED_V8_CANDIDATE,
        'cache_signature': CACHE_SIGNATURE,
        'fold_assignment_sha256': FOLD_SIGNATURE,
        'model_id': MODEL_ID,
        'model_revision': MODEL_REVISION,
        'selected_approach': APPROACH_DECISION['selected_family'],
        'fixed_hyperparameters': LOCKED_V8_CONTRACT['training'],
        'conditions': confirmation_conditions,
        'gates': CONFIRMATION_GATES,
        'observed': {
            'aggregate_macro_f1': confirmation_candidate_metrics['macro_f1'],
            'aggregate_control_macro_f1': confirmation_control_metrics['macro_f1'],
            'folds_beating_control': folds_beating_control,
            'candidate_ro_errors': candidate_ro_errors,
            'verified_v7_ro_errors': EXPECTED_V7_CONFIRMATION_RO_ERRORS,
            'candidate_electronic_f1': candidate_electronic_f1,
            'control_electronic_f1': control_electronic_f1,
            'candidate_total_errors': confirmation_candidate_metrics['total_errors'],
            'control_total_errors': confirmation_control_metrics['total_errors'],
        },
        'historical_assessment_folds_reused_for_confirmation': list(CONFIRMATION_FOLDS),
        'official_test_accessed': False,
    }
    FINAL_ADOPTION_DECISION = {
        'schema_version': V8_SCHEMA_VERSION,
        'revision_marker': V8_REVISION_MARKER,
        'adopt_v8': confirmation_pass,
        'final_pipeline': SELECTED_V8_CANDIDATE if confirmation_pass else 'V7 regularized_mlp_final',
        'reason': 'V8 passed every locked fold-0 and cross-fold confirmation gate.' if confirmation_pass else 'V8 passed fold 0 but failed at least one locked cross-fold confirmation gate; retain V7.',
        'fold0_decision': FOLD0_DECISION,
        'confirmation': CONFIRMATION_STATUS,
        'cache_signature': CACHE_SIGNATURE,
        'fold_assignment_sha256': FOLD_SIGNATURE,
        'model_id': MODEL_ID,
        'model_revision': MODEL_REVISION,
        'selected_approach': APPROACH_DECISION['selected_family'],
        'fixed_hyperparameters': LOCKED_V8_CONTRACT['training'],
        'gates': {'fold0': FOLD0_GATES, 'confirmation': CONFIRMATION_GATES},
        'official_test_accessed': False,
        'submission_created': False,
    }
    write_json_atomic(FINAL_ADOPTION_DECISION, ARTIFACT_DIR / 'final_adoption_decision.json')
else:
    write_json_atomic(CONFIRMATION_STATUS, ARTIFACT_DIR / 'confirmation_status.json')

print('Confirmation status:', json.dumps(CONFIRMATION_STATUS, indent=2))

Confirmation status: {
  "ran": false,
  "reason": "Fold-0 gate did not pass.",
  "cache_signature": "c722dc39a8cba286d3aba34d88f3e8f87071353bc6585e755873659de8560ddc",
  "fold_assignment_sha256": "0a06976e5dc3cdb0b3f1f5043ff033bf95a40dd2b0cb82154bbf34d701ee305c",
  "model_id": "facebook/dinov3-vit7b16-pretrain-lvd1689m",
  "model_revision": "b80367753773648a6793235ab9c65cdbb029506f",
  "selected_approach": "binary Recyclable/Organic specialist with Electronic-preserving hierarchical probability-mass fusion",
  "fixed_hyperparameters": {
    "seed": 42,
    "max_epochs": 15,
    "patience": 5,
    "optimizer": "AdamW",
    "learning_rate": 0.0008,
    "weight_decay": 0.0001,
    "scheduler": "CosineAnnealingLR",
    "batch_size": 256,
    "gradient_clip_max_norm": 1.0
  },
  "gates": {
    "minimum_aggregate_macro_f1": 0.9905,
    "minimum_folds_beating_reproduced_control": 3,
    "maximum_ro_errors_exclusive": 187,
    "minimum_electronic_f1": 0.9933,
    "maximum_electronic_f1_degrad

## 12. Final adoption decision

Adoption requires every predeclared fold-0 gate and, when reached, every cross-fold confirmation gate. A scientific rejection is a clean result: V7 remains the final pipeline. No leaderboard or official-test feedback enters this decision.

## 13. Artifact inventory and limitations

The final manifest records cache/model/fold provenance, approach, fixed hyperparameters, gates, decisions, compliance, and all generated files. Runtime success, environment compatibility, and exact control reproducibility can only be established when this unexecuted notebook runs on Kaggle with the complete V7 artifact directory.

In [10]:
BASE_REQUIRED_ARTIFACTS = [
    'v7_artifact_audit.json',
    'cache_compatibility_audit.json',
    'approach_decision.json',
    'locked_v8_contract.json',
    'fold_label_audit.json',
    'control_reproduction.json',
    'fold0_candidate_comparison.csv',
    'fold0_candidate_metrics.json',
    'fold0_paired_error_delta.csv',
    'fold0_corrected_samples.csv',
    'fold0_regressed_samples.csv',
    'fold0_high_confidence_errors.csv',
    'fold0_prediction_disagreement_matrix.csv',
    'fold0_control_confidence_bins.csv',
    'fold0_diagnostic_summary.json',
    'fold0_confusion_comparison.png',
    'fold0_confidence_margin_comparison.png',
    'fold0_decision.json',
    'final_adoption_decision.json',
]
head_artifact_suffixes = ('_history.csv', '_classification_report.csv', '_predictions.csv', '_metrics.json', '.pt')
fold0_head_stems = [
    'fold0_control_{}_fold{}_seed{}'.format(CONTROL_NAME, DEVELOPMENT_FOLD, SEED),
    'fold0_specialist_{}_fold{}_seed{}'.format(SPECIALIST_NAME, DEVELOPMENT_FOLD, SEED),
] + ['fold0_{}_fold{}_seed{}'.format(name, DEVELOPMENT_FOLD, SEED) for name in FUSION_SPECS]
for stem in fold0_head_stems:
    BASE_REQUIRED_ARTIFACTS.extend('heads/' + stem + suffix for suffix in head_artifact_suffixes)
if CONFIRMATION_RAN:
    BASE_REQUIRED_ARTIFACTS.extend([
        'confirmation_per_fold_comparison.csv',
        'confirmation_aggregate_comparison.csv',
        'confirmation_selected_predictions.csv',
        'confirmation_aggregate_metrics.json',
        'confirmation_paired_error_delta.csv',
        'confirmation_corrected_samples.csv',
        'confirmation_regressed_samples.csv',
        'confirmation_high_confidence_errors.csv',
        'confirmation_prediction_disagreement_matrix.csv',
        'confirmation_control_confidence_bins.csv',
        'confirmation_diagnostic_summary.json',
        'confirmation_confusion_comparison.png',
        'confirmation_confidence_margin_comparison.png',
    ])
    for fold in CONFIRMATION_FOLDS:
        confirmation_head_stems = (
            'confirmation_control_{}_fold{}_seed{}'.format(CONTROL_NAME, fold, SEED),
            'confirmation_specialist_{}_fold{}_seed{}'.format(SPECIALIST_NAME, fold, SEED),
            'confirmation_{}_fold{}_seed{}'.format(SELECTED_V8_CANDIDATE, fold, SEED),
        )
        for stem in confirmation_head_stems:
            BASE_REQUIRED_ARTIFACTS.extend('heads/' + stem + suffix for suffix in head_artifact_suffixes)
        prefix = 'confirmation_fold{}'.format(fold)
        BASE_REQUIRED_ARTIFACTS.extend(prefix + suffix for suffix in (
            '_paired_error_delta.csv', '_corrected_samples.csv', '_regressed_samples.csv',
            '_high_confidence_errors.csv', '_prediction_disagreement_matrix.csv',
            '_control_confidence_bins.csv', '_diagnostic_summary.json',
            '_confusion_comparison.png', '_confidence_margin_comparison.png',
        ))
else:
    BASE_REQUIRED_ARTIFACTS.append('confirmation_status.json')
missing_artifacts = [name for name in BASE_REQUIRED_ARTIFACTS if not (ARTIFACT_DIR / name).is_file()]
require(not missing_artifacts, 'Required V8 artifacts are missing: {}.'.format(missing_artifacts))

RUN_MANIFEST = {
    'schema_version': V8_SCHEMA_VERSION,
    'revision_marker': V8_REVISION_MARKER,
    'experiment_name': EXPERIMENT_NAME,
    'selected_approach': APPROACH_DECISION['selected_family'],
    'approach_decision': APPROACH_DECISION,
    'locked_v8_contract': LOCKED_V8_CONTRACT,
    'locked_v8_contract_signature': LOCKED_V8_CONTRACT_SIGNATURE,
    'v7_root': str(V7_ROOT),
    'v7_cache_signature': CACHE_SIGNATURE,
    'fold_assignment_sha256': FOLD_SIGNATURE,
    'model_id_inherited_from_v7': MODEL_ID,
    'model_revision_inherited_from_v7': MODEL_REVISION,
    'backbone_state': 'fully frozen in V7; not imported, loaded, instantiated, or forwarded in V8',
    'control_reproduction': CONTROL_REPRODUCTION,
    'fold0_decision': FOLD0_DECISION,
    'confirmation_status': CONFIRMATION_STATUS,
    'final_adoption_decision': FINAL_ADOPTION_DECISION,
    'fixed_hyperparameters': LOCKED_V8_CONTRACT['training'],
    'gates': {'fold0': FOLD0_GATES, 'confirmation': CONFIRMATION_GATES},
    'environment': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'device': str(DEVICE),
        'cuda_available': torch.cuda.is_available(),
    },
    'competition_compliance': {
        'official_labeled_train_only': True,
        'visual_cached_features_only': True,
        'public_pretrained_model_documented': MODEL_ID,
        'external_labeled_images_used': False,
        'test_inspected_or_enumerated': False,
        'test_hashed_or_decoded': False,
        'test_embedding_or_inference': False,
        'test_tuning_or_calibration': False,
        'template_accessed': False,
        'submission_created': False,
        'filenames_paths_metadata_exif_dimensions_used_as_predictive_features': False,
        'pseudo_labeling': False,
        'mass_relabeling': False,
        'difficult_samples_removed_from_model_disagreement': False,
        'test_access_guard_active': True,
    },
    'limitations': [
        'Folds 1-4 informed the V8 direction and are historical assessment folds reused for confirmation, not pristine test-like data.',
        'The image gallery suggests ambiguity; suspicious rows are manual-review candidates only and official labels remain unchanged.',
        'The notebook must reproduce V7 within tolerance at runtime before any V8 screening.',
        'No local or Kaggle runtime success is implied by the static notebook artifact.',
    ],
    'notebook_elapsed_seconds': time.perf_counter() - NOTEBOOK_STARTED_AT,
}
write_json_atomic(RUN_MANIFEST, ARTIFACT_DIR / 'run_manifest.json')
artifact_inventory = sorted(str(path.relative_to(ARTIFACT_DIR)) for path in ARTIFACT_DIR.rglob('*') if path.is_file())
RUN_MANIFEST['artifact_inventory'] = artifact_inventory
RUN_MANIFEST['artifact_count'] = len(artifact_inventory)
write_json_atomic(RUN_MANIFEST, ARTIFACT_DIR / 'run_manifest.json')
print(json.dumps(FINAL_ADOPTION_DECISION, indent=2))
print('V8 complete. Train-only cache experiment artifacts:', ARTIFACT_DIR)

{
  "schema_version": 1,
  "revision_marker": "v8_ro_hierarchical_specialist_final_cache_v1",
  "adopt_v8": false,
  "final_pipeline": "V7 regularized_mlp_final",
  "reason": "No locked specialist fusion passed every fold-0 gate; stop cleanly and retain V7.",
  "fold0_decision": {
    "schema_version": 1,
    "revision_marker": "v8_ro_hierarchical_specialist_final_cache_v1",
    "pass": false,
    "selected_candidate": null,
    "diagnostic_candidate": "ro_specialist_full",
    "candidate_ranking": [
      "ro_specialist_full",
      "ro_specialist_blend75"
    ],
    "gate_results": {
      "ro_specialist_blend75": {
        "pass": false,
        "conditions": {
          "macro_f1": false,
          "gain_over_control": false,
          "ro_error_reduction": false,
          "total_errors": false,
          "electronic_f1": true,
          "corrected_samples": true,
          "not_single_sample_dependent": false,
          "locked_contract_unchanged": true,
          "electronic_dec